Version Date: March 16, 2026

## Author: Sandy G Cabanes
## Role: Sole Contributor (Volunteer)
## Date: March 2026
## Response Dates: September 2025 to March 2026
## Licensed Under MIT

# 0. Imports and Directories

In [1]:
# 0. Imports & Config
# ~~~~~~~~~~~~~~~~~~~~~~~~~
import pandas as pd
import unicodedata
from pathlib import Path
import numpy as np
import csv # used in apply_lookup function
import datetime
import os

In [2]:
# 0. Create Directories
# Store all lookup files for single-response columns and exploded multi-response csv files
lookup_dir = Path("lookup_dir")
lookup_dir.mkdir(exist_ok=True)

# Store all parquet files
parquet_outputs_dir = Path("parquet_outputs_dir")
parquet_outputs_dir.mkdir(exist_ok=True)

# Store all the outputs as csv so cleaning can be split into modules
csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)

# Copy of all the final outputs for data model
final_outputs_dir = Path("final_outputs_dir")
final_outputs_dir.mkdir(exist_ok=True)

# Store preliminary unique values
unique_dir = Path("unique_dir")
unique_dir.mkdir(exist_ok=True)

# Store all csv and map files using country and city columns
location_dir = Path("location_dir")
location_dir.mkdir(exist_ok = True)

# Store all datamart duckdb sqlite and for_tableau.xlsx 
data_mart= Path("data_mart")
data_mart.mkdir(exist_ok = True)


# Lineage ← →

- Raw file  
  - csv_raw.csv → df_raw
  - df_raw (cleaned) ← df_raw

- Single‑response family  
  - df_single_no_grps ← df_raw[single_cols_no_grps].copy()  
  - df_single_with_grps ← df_single_no_grps.copy()  
  - df_possible_dups ← df_likely_dups that have a score == 1
  - df_sim_raw ← df_single_no_grps.copy()
  - df_sim_sorted ← df_sim_raw

- Multi‑response family  
  - csv_raw.csv → df_raw_multi → clean MS Fabric, clean and standardize None responses
  - df_multi ← df_raw_multi[["resp_id"] + multi_response_cols].copy()
  - datamasters_count ← df_multi_copy
  - df_exploded = explode_and_lookup(df_raw, col) → saved to csv as {col}_exploded.csv
  - df_exploded_cleaned 
  - df_online ← attended_online_exploded_cleaned.csv, themed
  - df_merged_dups = df_possible_duplicates.merge()

- Location family  
  - df_loc_new ← df_single_with_groups
  - df_ph_grp, df_non_ph_grp ← df_loc_new 
  - overseas ← df_ph_grp 
  - df_ph_geo, df_non_ph_geo ← df_ph_grp, df_non_ph_grp
  - df_ph_geo_clean ← df_ph_geo
  - df_non_ph_geo_clean ← df_non_ph_geo
  - df_all_geo_clean ← df_ph_geo_clean, df_non_ph_geo_clean

-  Data mart family  
    - survey2026.duckdb
    - survey.sqlite
    - for_tableau.xls


# REMOVE TEST RUN FIRST!!!

In [16]:
# 1. Load Raw Data and Rename Columns (Raw -> Transformed)
# IMPORTANT: Remove the test runs first before saving as csv_raw.csv
df_raw= pd.read_csv(
    "csv_raw.csv",
    sep=",",
    quotechar='"',
    engine="python",
    encoding="utf-8-sig"
)
# Drop test runs
df_raw = df_raw[df_raw['Email (Optional)  '] != 'email@testrun.com']

# Drop timestamp, email, and end of survey
df_raw = df_raw.drop(columns = ['Timestamp', 'Email (Optional)  ', 'End of survey'])


In [17]:
# 1a. Check the headers
headers_raw = df_raw.columns.to_list()
headers_raw

['Country of Residence, current',
 "PHILIPPINE City of Residence, current (Please scroll to see all options.  If municipality, please write in Other - ______.   If not in the Philippines, please choose 'Not applicable' .)",
 'NON-PHILIPPINE City of Residence (Choose Not Applicable if you are currently in the Philippines)',
 'Gender',
 'Age (Please enter a number)',
 'Latest education status',
 'Do you have a Computer or Data-Related degree? ',
 'If you have a Computer-Related degree, or Data-Related degree,  please indicate the degree below. \n(Otherwise, please choose Not Applicable, first button.)',
 'Choose the digital learning platforms or tools you are currently using for data-related learning:',
 'What best describes the CURRENT STAGE of your data career?',
 'Thinking of your most recent job search, which platform or method gave you the most success?',
 'Type of employer (Please select one option that best describes your current employment situation.)',
 'What Industry currently 

In [18]:
# 1b. Rename Columns (Raw → Transformed)
# columns = rename_map dictionary

rename_map = {
    "Country of Residence, current" : "country", #new
    "City of Residence, current": "city",
    "PHILIPPINE City of Residence, current (Please scroll to see all options.  If municipality, please write in Other - ______.   If not in the Philippines, please choose 'Not applicable' .)" : "city_ph",
    'NON-PHILIPPINE City of Residence (Choose Not Applicable if you are currently in the Philippines)' : "city_non_ph",
    "Gender": "gender",
    "Age (Please enter a number)": "age",
    "Latest education status": "educstat",
    "Do you have a Computer or Data-Related degree? ": "have_comp_degree", #new
    "If you have a Computer-Related degree, or Data-Related degree,  please indicate the degree below. \n(Otherwise, please choose Not Applicable, first button.)" : "comp_related_degree", #new
    "Choose the digital learning platforms or tools you are currently using for data-related learning:": "digitools",
    "What best describes the CURRENT STAGE of your data career?" : "careerstg",
    "Thinking of your most recent job search, which platform or method gave you the most success?": "successmethod",
    "Type of employer (Please select one option that best describes your current employment situation.)" : "employertype",
    "What Industry currently working in:": "industry",
    "Monthly Salary Range (Or monthly income from main source) If paid per project, please approximate monthly equivalent." : "salary",
    "How long have you been in the current salary range or monthly income range?" : "how_long_in_salary",  #new
    "What is your current Type of Work?": "typework", #new 
    "What is your current Work Site situation?": "sitework",
    "If working and in the Philippines, what is your working shift schedule most of the time? (If not in the Philippines, choose Not Applicable.)" : "shift_schedule",  #new
    "What best describes MAJORITY of your day-to-day role? (More than 50% of your role)": "datarole", #uncategorized in 2024
    "What other descriptions comprise the REST of your role? (Click all that apply)": "restofrole", #uncategorized in 2024
    "Thinking of the past 12 months in terms of side gigs or side hustles, which one applies to you?" : "side_gig", #new
    "On a scale of 1 to 10, how satisfied are you with your current data-related role or data career?  (1 - Not Satisfied At All, 10 - Extremely Satisfied)" : "satisfaction", #new
    "What is the size of your Data Team?": "sizeteam",
    "What are the data INGESTION tools you currently use? ": "ingestion",
    "What are the data TRANSFORMATION tools you currently use?  ": "transform",
    "What are the DATA LAKES OR OBJECT STORAGE SERVICES you currently use?" : "storage", #new
    "What are the data WAREHOUSES or RELATIONAL DATABASES that you currently use?   ": "warehs",
    "What are the data ORCHESTRATION tools you currently use?  ": "orchest",
    "What are the BUSINESS INTELLIGENCE tools you currently use? ": "busint",
    "What are the CLOUD-NATIVE DATA PLATFORMS you currently use?" : "cloudplat",
    "Which of the following \nNON-CODING-LANGUAGE DATA TOOLS OR PRODUCTIVITY TOOLS \ndo you use on a regular basis? Choose all that apply.": "generaltools",
    "Which of the following CODING LANGUAGES OR TOOLS do you use on a regular basis? Choose all that apply.": "whatused",
    "Thinking of WORK, do you currently use AI in your work?": "ai_work", #new
    "Thinking of STUDIES or UPSKILLING, do you currently use AI for study?": "ai_study", #new
    "If you answered Yes to use of AI at work or for study, please choose which AI tools you currently use.  (Choose Not Applicable if you do not use AI.)" : "ai_tools", # revised
    "What hardware do you currently use for data?": "hardware",
    "Which of the following DEP initiatives and channels are you aware of? Choose all that apply." : "dep_init", #revised
    "Any specific tasks, skills, knowledge or resources you are willing to contribute to the group as of today?": "volunteer",
    "Any specific needs you are trying to address by joining DEP Facebook group?" : "spneeds",
    "Thinking of data-related communities, what other Facebook communities do you follow?": "otherfb",
    'In the Past 12 months, what was the most recent DEP or data-related meet-up, hackathon, convention, that you attended IN PERSON? (Maximum 50 characters.  If none, it\'s ok, you can type "None")': "attended_inperson", #new
    'In the Past 12 months, what was the most recent DEP or data-related activity that you attended ONLINE? (E.g. discord event, free webinar, etc.   Maximum 50 characters.  If none, it\'s ok, you can type "None")' : "attended_online" #new
}


df_raw.rename(columns=rename_map, inplace=True)

In [19]:
#1c. Check mapped headers, Data Wrangler
headers_revised= df_raw.columns.to_list()
headers_revised

['country',
 'city_ph',
 'city_non_ph',
 'gender',
 'age',
 'educstat',
 'have_comp_degree',
 'comp_related_degree',
 'digitools',
 'careerstg',
 'successmethod',
 'employertype',
 'industry',
 'salary',
 'how_long_in_salary',
 'typework',
 'sitework',
 'shift_schedule',
 'datarole',
 'restofrole',
 'satisfaction',
 'side_gig',
 'sizeteam',
 'ingestion',
 'transform',
 'storage',
 'warehs',
 'orchest',
 'busint',
 'cloudplat',
 'generaltools',
 'whatused',
 'ai_work',
 'ai_study',
 'ai_tools',
 'hardware',
 'dep_init',
 'volunteer',
 'spneeds',
 'otherfb',
 'attended_online',
 'attended_inperson']

In [23]:
#1d. Column data types
print(df_raw.info())

print ("~"*50)  
print ("Result as of: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print ("Total respondents: ", df_raw.shape[0])
print ("Total columns: ", df_raw.shape[1])
print ("Total working: ", df_raw["successmethod"].count())
print ("~"*50)

<class 'pandas.core.frame.DataFrame'>
Index: 1861 entries, 1 to 1861
Data columns (total 42 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   country              1861 non-null   object 
 1   city_ph              1861 non-null   object 
 2   city_non_ph          1844 non-null   object 
 3   gender               1861 non-null   object 
 4   age                  1861 non-null   int64  
 5   educstat             1861 non-null   object 
 6   have_comp_degree     1861 non-null   object 
 7   comp_related_degree  1858 non-null   object 
 8   digitools            1860 non-null   object 
 9   careerstg            1861 non-null   object 
 10  successmethod        808 non-null    object 
 11  employertype         808 non-null    object 
 12  industry             808 non-null    object 
 13  salary               808 non-null    object 
 14  how_long_in_salary   808 non-null    object 
 15  typework             808 non-null    object

In [26]:
# 2. First Global Cleaning (Inline)
# Fillna, normalize font, strip, collapse spaces, title-case specific columns, 
# Replace Other with blanks, replace 0 age with min age, cap at 95

# Fillna, normalize font ex. ñ, strip, split
for col in df_raw.select_dtypes(include="object").columns:
    # Normalize, strip, collapse spaces
    df_raw[col] = (
        df_raw[col]
        .fillna("Not Applicable")
        .astype(str)
        .str.replace(r'[\u2010-\u2015]', '-', regex=True)
        .str.replace("â€“", "-", regex=False)
        .apply(lambda v: " ".join(
            unicodedata.normalize("NFKC", v.replace("ñ", "n")).strip().split()
        ))
    )
    # Title-case specific columns (For UK, KSA, etc. handle in the lookup portion to revert)
    if col in ["country", "city_ph", "city_non_ph", "datarole", "rest_of_role", "industry"]:
        df_raw[col] = df_raw[col].str.title()

# Replace strings that start with 'Other' with blanks
for col in df_raw.select_dtypes(include="object").columns:
     df_raw[col] = df_raw[col].apply(lambda v: "" if v.startswith("Other") else v)

# Replace age of 0 with min age, and cap at 95
valid_min_age = df_raw.loc[df_raw["age"] > 0, "age"].min()
df_raw["age"] = df_raw["age"].replace(0, valid_min_age)

max_age = df_raw["age"].max()
df_raw["age"] = df_raw["age"].apply(lambda x: max_age if x > 95 else x)

print ("First global cleaning done.")
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


First global cleaning done.
Date and Time:  2026-03-16 17:13:30


In [27]:
#2a. Check after first global cleaning Or Open in Data Wrangler
df_raw.head()

,country,city_ph,city_non_ph,gender,age,educstat,have_comp_degree,comp_related_degree,digitools,careerstg,...,ai_work,ai_study,ai_tools,hardware,dep_init,volunteer,spneeds,otherfb,attended_online,attended_inperson
1,Philippines,Pampanga,Not Applicable,Male,23,Bachelor's or College Degree,No computer-related or data-related degree,Not Applicable,Coursera;Datacamp;Youtube;Eskwelabs,Career shifter - currently working in a non-da...,...,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",Chatgpt,desktop;laptop,DEP DataCamp scholarship;DEP DataMasters;DEP T...,Answer surveys;Give advice to people in the gr...,Be part of a community;Career advancement;Cont...,None of the above,Eskwelabs bootcamp and datamasters,Not Applicable
2,Philippines,Binan,Not Applicable,Male,36,Master's degree,No computer-related or data-related degree,Not Applicable,Datacamp;MicrosoftLearn;W3Schools;Youtube,Professional - employed full-time in a data-re...,...,"Yes, I use AI tools WEEKLY for work.","Yes, I use AI tools WEEKLY for study",Chatgpt;Copilot,laptop,DEP DataCamp scholarship;DEP The Puso Project;...,Answer surveys;Give advice to people in the gr...,Be part of a community;Career advancement;Cont...,Analytics Association of the Philippines;Commu...,Career Shifting and Data Analysis discord meeting,Not Applicable
3,Philippines,Mandaluyong,Not Applicable,Female,32,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Electronics and Communications Engineering,Cisco;Coursera;Datacamp;LinkedIn Learning;O'Re...,Professional - employed full-time in a data-re...,...,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools WEEKLY for study",Chatgpt;Copilot;Gemini;Grok;Perplexity,laptop;VM,DEP DataCamp scholarship;DEP DataHub;DEP Study...,Answer surveys;Sponsor/Invest,Continued learning,Data Engineering Pilipinas,Not Applicable,Not Applicable
4,Philippines,San Quintin,Not Applicable,Male,34,Bachelor's or College Degree,No computer-related or data-related degree,Not Applicable,Cisco;Coursera;Datacamp;Udemy;W3Schools;Youtube,Career shifter - currently working in a non-da...,...,"No, but I plan to soon.","Yes, I use AI tools DAILY for study.",Chatgpt;Cici;Copilot,desktop;laptop,DEP DataCamp scholarship;DEP youtube channel,Answer surveys;Share knowledge;Teach soft skills,Be part of a community;Career advancement;Cont...,Datasense Philippines,Not Applicable,None
5,Philippines,Taguig,Not Applicable,Male,29,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Applied Mathematics or Related,AWS Educate;Coursera;Udemy,Professional - employed full-time in a data-re...,...,"Yes, I use AI tools DAILY for work.","No, but I plan to soon.",Chatgpt;Claude;Copilot;Cursor;Gemini;Github co...,laptop;VM;workstation,DEP DataCamp scholarship;DEP DataMasters;DEP d...,Answer surveys;Give advice to people in the gr...,Be part of a community;Continued learning;Network,Analytics Association of the Philippines;AWS U...,discord event,Not Applicable


In [28]:
# 3. Add resp_id and resp_id_rand to df_raw after first global cleaning
# Add resp_id
if "resp_id" not in df_raw.columns:
    df_raw.insert(0, "resp_id", range(1, len(df_raw) + 1))

# Add resp_id_rand using secrets
import secrets

def generate_ids(n):
    ids = set()
    # Running the loop until the set contains the required number of unique IDs
    while len(ids) < n:
        # Creating a random 6-character hex string
        new_id = f"2025-{secrets.token_hex(3)}"
        ids.add(new_id)
    return list(ids)

# Assigning the unique list to the dataframe column
df_raw['resp_id_rand'] = generate_ids(len(df_raw))

In [29]:
#3a. Check in Data Wrangler → export as df_raw.parquet
df_raw.head()

,resp_id,country,city_ph,city_non_ph,gender,age,educstat,have_comp_degree,comp_related_degree,digitools,...,ai_study,ai_tools,hardware,dep_init,volunteer,spneeds,otherfb,attended_online,attended_inperson,resp_id_rand
1,1,Philippines,Pampanga,Not Applicable,Male,23,Bachelor's or College Degree,No computer-related or data-related degree,Not Applicable,Coursera;Datacamp;Youtube;Eskwelabs,...,"Yes, I use AI tools DAILY for study.",Chatgpt,desktop;laptop,DEP DataCamp scholarship;DEP DataMasters;DEP T...,Answer surveys;Give advice to people in the gr...,Be part of a community;Career advancement;Cont...,None of the above,Eskwelabs bootcamp and datamasters,Not Applicable,2025-eed25f
2,2,Philippines,Binan,Not Applicable,Male,36,Master's degree,No computer-related or data-related degree,Not Applicable,Datacamp;MicrosoftLearn;W3Schools;Youtube,...,"Yes, I use AI tools WEEKLY for study",Chatgpt;Copilot,laptop,DEP DataCamp scholarship;DEP The Puso Project;...,Answer surveys;Give advice to people in the gr...,Be part of a community;Career advancement;Cont...,Analytics Association of the Philippines;Commu...,Career Shifting and Data Analysis discord meeting,Not Applicable,2025-c107bd
3,3,Philippines,Mandaluyong,Not Applicable,Female,32,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Electronics and Communications Engineering,Cisco;Coursera;Datacamp;LinkedIn Learning;O'Re...,...,"Yes, I use AI tools WEEKLY for study",Chatgpt;Copilot;Gemini;Grok;Perplexity,laptop;VM,DEP DataCamp scholarship;DEP DataHub;DEP Study...,Answer surveys;Sponsor/Invest,Continued learning,Data Engineering Pilipinas,Not Applicable,Not Applicable,2025-b053b8
4,4,Philippines,San Quintin,Not Applicable,Male,34,Bachelor's or College Degree,No computer-related or data-related degree,Not Applicable,Cisco;Coursera;Datacamp;Udemy;W3Schools;Youtube,...,"Yes, I use AI tools DAILY for study.",Chatgpt;Cici;Copilot,desktop;laptop,DEP DataCamp scholarship;DEP youtube channel,Answer surveys;Share knowledge;Teach soft skills,Be part of a community;Career advancement;Cont...,Datasense Philippines,Not Applicable,None,2025-a03238
5,5,Philippines,Taguig,Not Applicable,Male,29,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Applied Mathematics or Related,AWS Educate;Coursera;Udemy,...,"No, but I plan to soon.",Chatgpt;Claude;Copilot;Cursor;Gemini;Github co...,laptop;VM;workstation,DEP DataCamp scholarship;DEP DataMasters;DEP d...,Answer surveys;Give advice to people in the gr...,Be part of a community;Continued learning;Network,Analytics Association of the Philippines;AWS U...,discord event,Not Applicable,2025-ae4e76


In [30]:
#3a.1 Export df_raw as parquet file
df_raw.to_parquet(parquet_outputs_dir/ "df_raw.parquet", index=False)
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("df_raw.parquet created. Global cleaning done.")

Date and Time: 2026-03-16 17:14:05
df_raw.parquet created. Global cleaning done.


In [ ]:
#3b. Count empty strings and null values or isna and print only those with values
# empty_counts series: Count empty strings - this will include the "" replacements above
empty_counts = (df_raw == "").sum()
empty_columns_only = empty_counts[empty_counts > 0] #only the ones with values
print("\n\nEmpty strings", "~"*20)
if not empty_columns_only.empty:
    print(empty_columns_only)
else:
    print("No empty strings found in any column.")

# nan_counts series: Count NaN / Null values -- expect satisfaction to be total n minus working n
nan_counts = df_raw.isna().sum()
nan_columns_only = nan_counts[nan_counts > 0] #only the ones with values
print("\n\nNaN / Null values", "~"*20)
if not nan_columns_only.empty:
    print(nan_columns_only)
else:
    print("No NaN values found in any column.")

# Calculate NaN and empty cells 
combined_counts = df_raw.apply(
    lambda col: (
        col.isna() | 
        (col.astype(str).str.strip() == "")
    ).sum()
)
combined_only  = combined_counts[combined_counts > 0] #only the ones with values
print("\n\nCombined NaN or blanks", "~"*20)
if not combined_only.empty:
    print(combined_only)





Empty strings ~~~~~~~~~~~~~~~~~~~~
datarole      31
restofrole    31
transform      2
dtype: int64


NaN / Null values ~~~~~~~~~~~~~~~~~~~~
satisfaction    1053
dtype: int64


Combined NaN or blanks ~~~~~~~~~~~~~~~~~~~~
datarole          31
restofrole        31
satisfaction    1053
transform          2
dtype: int64


In [ ]:
# 3c. Replace all empty strings and null values with "Not Applicable"
# Clean whitespace and empty strings
# Re-import from checkpoint. Normalization re-applied as a defensive
# guard — CSV round-trips can reintroduce whitespace and mixed dtypes.
df_raw = df_raw.replace(r'^\s*$', "Not Applicable", regex=True)

# Clean technical Nulls (NaN)
df_raw = df_raw.fillna("Not Applicable")

# Special treatment for satisfaction, that requires numeric values
df_raw["satisfaction"] = df_raw["satisfaction"].apply(lambda x: 999 if x == "Not Applicable" else x)

In [ ]:
df_raw

In [ ]:
# 3d. If country is not Philippines, check if city_non_ph is Not Applicable or a Philippine city

df_nonph =df_raw[df_raw['country'] != 'Philippines'][['resp_id', 'country', 'city_non_ph']]
nonphlist =sorted(list(df_nonph['city_non_ph'].unique()))
print("Before cleaning non ph cities", "~"*20)
print (nonphlist)

# Ok, nonph cities are not Philippine cities.  

# Change the 'Not Applicable' to 'Prefer Not To Say'
filter_nonph = (df_raw['country'] != 'Philippines') & (df_raw['city_non_ph'] == 'Not Applicable')
df_raw.loc[filter_nonph, 'city_non_ph'] = 'Prefer Not To Say'

# Check again
df_nonph =df_raw[df_raw['country'] != 'Philippines'][['resp_id', 'country', 'city_non_ph']]
nonphlist =sorted(list(df_nonph['city_non_ph'].unique()))
print("After cleaning non ph cities", "~"*20)
print (nonphlist)

Before cleaning non ph cities ~~~~~~~~~~~~~~~~~~~~
['Abu Dhabi, United Arab Emirates', 'Accra', 'Alexandria', 'Amman', 'Bangkok', 'Barishal', 'Benin City', 'Cairo', 'Cape Town', 'Cerro De Pasco', 'Cheongju City', 'Culiacan', 'Diplo', 'Douala', 'Dubai', 'Ho Chi Minh', 'Jeddah', 'Kansas', 'Kathmandu', 'Kinshasa', 'Ksa', 'Kuala Lumpur', 'Lagos', 'Lagos State', 'London', 'Nairobi', 'Newcastle', 'Nigeria', 'Not Applicable', 'Oslo', 'Pasco', 'Phnom Penh', 'Prefer Not To Say', 'Pretoria', 'Regina', 'Riyadh', "Sana'A", 'Soroti City Uganda', 'Sydney', 'Uae', 'Ughelli']
After cleaning non ph cities ~~~~~~~~~~~~~~~~~~~~
['Abu Dhabi, United Arab Emirates', 'Accra', 'Alexandria', 'Amman', 'Bangkok', 'Barishal', 'Benin City', 'Cairo', 'Cape Town', 'Cerro De Pasco', 'Cheongju City', 'Culiacan', 'Diplo', 'Douala', 'Dubai', 'Ho Chi Minh', 'Jeddah', 'Kansas', 'Kathmandu', 'Kinshasa', 'Ksa', 'Kuala Lumpur', 'Lagos', 'Lagos State', 'London', 'Nairobi', 'Newcastle', 'Nigeria', 'Oslo', 'Pasco', 'Phnom Penh'

In [ ]:
# Open in Data Wrangler
df_nonph

In [ ]:
# 3e. If country is Philippines, check 
df_ph = df_raw[df_raw['country'] == 'Philippines'][['resp_id', 'country', 'city_ph', 'city_non_ph']]
df_ph # Open in Data Wrangler

# 3f. Human edits to specific respondents
- Specific resp_id with country == "Philippines" who put the city_ph in city_non_ph
- Specific resp_id with country != "Philippines" who answered city_ph instead of skipping

"""
```
Specific resp_id with country == "Philippines" who put the city_ph in city_non_ph
Edit to the following final values for city_ph and city_non_ph
resp_id, country, city_ph, city_non_ph
409, Philippines, Tuguegarao City, Not Applicable
436, Philippines, Dasmarinas City, Not Applicable
561, Philippines, Gen Mariano Alvarez, Not Applicable
856, Philippines, Calamba, Not Applicable
1149, Philippines, San Ildefonso Bulacan, Not Applicable
1502, Philippines, Calamba, Not Applicable
1806, Philippines, Sibulan, Not Applicable

Specific resp_id with country != "Philippines" who answered city_ph instead of skipping
Decision: Preserve current answers.  Filter only during the region_ph assignment section.
resp_id, country, city_ph
1086,KSA,Bulacan
742,Canada,Caloocan
1005,UAE,Aleosan
```
"""


In [44]:
# 3f. Special edits for some city_ph answers

# Convert above into a lookup dictionary
city_updates = {
    409: ["Tuguegarao City", "Not Applicable"],
    436: ["Dasmarinas City", "Not Applicable"],
    561: ["Gen Mariano Alvarez", "Not Applicable"],
    856: ["Calamba", "Not Applicable"],
    1149: ["San Ildefonso Bulacan", "Not Applicable"],
    1502: ["Calamba", "Not Applicable"],
    1806: ["Sibulan", "Not Applicable"]
}

# Update the table by matching the identification numbers
for resp_id, values in city_updates.items():
    df_ph.loc[df_ph["resp_id"] == resp_id, ["city_ph", "city_non_ph"]] = values
df_ph[df_ph["resp_id"].isin(city_updates.keys())]


,resp_id,country,city_ph,city_non_ph
409,409,Philippines,Tuguegarao City,Not Applicable
436,436,Philippines,Dasmarinas City,Not Applicable
561,561,Philippines,Gen Mariano Alvarez,Not Applicable
856,856,Philippines,Calamba,Not Applicable
1149,1149,Philippines,San Ildefonso Bulacan,Not Applicable
1502,1502,Philippines,Calamba,Not Applicable
1806,1806,Philippines,Sibulan,Not Applicable


In [46]:
# 3g. Convert the df_raw values as well
for resp_id, values in city_updates.items():
    df_raw.loc[df_raw["resp_id"] == resp_id, ["city_ph", "city_non_ph"]] = values
df_raw[df_raw["resp_id"].isin(city_updates.keys())][['resp_id', 'country', 'city_ph', 'city_non_ph']]


,resp_id,country,city_ph,city_non_ph
409,409,Philippines,Tuguegarao City,Not Applicable
436,436,Philippines,Dasmarinas City,Not Applicable
561,561,Philippines,Gen Mariano Alvarez,Not Applicable
856,856,Philippines,Calamba,Not Applicable
1149,1149,Philippines,San Ildefonso Bulacan,Not Applicable
1502,1502,Philippines,Calamba,Not Applicable
1806,1806,Philippines,Sibulan,Not Applicable


In [47]:
# 3c. Export df_raw to parquet_outputs_dir
df_raw.to_parquet(parquet_outputs_dir/ "df_raw.parquet", index=False)
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("df_raw.parquet created. Global cleaning done.")

Date and Time: 2026-03-16 17:53:06
df_raw.parquet created. Global cleaning done.


# 4. Handle single-response columns first
```
- single_cols_no_grps → single_cols_non_numeric → write unique_dir / "unique_single.txt"
- create starter csv for lookup with "raw" column → copy to lookup_dir → 
→ 1. direct clean 2. special edits 3. manual edits (hitl add "clean" column) 
```

In [ ]:
# 4a. Define single response columns for lookups
# This section uses df_single_no_grps_raw - 23 columns
# Later this will be split into three types of cleaning: direct, special and manual
single_cols_no_grps = [
    "resp_id", "country", "city_ph", "city_non_ph", "gender", "age", "educstat", "have_comp_degree", "comp_related_degree", "careerstg", "employertype", "industry", "salary", "how_long_in_salary", "typework", "sitework", "shift_schedule", "datarole", "side_gig", "satisfaction", "sizeteam", "ai_work", "ai_study", "resp_id_rand"
]

print ("Number of single cols:", len(single_cols_no_grps))
df_single_no_grps_raw = df_raw[single_cols_no_grps].copy()
df_single_no_grps_raw.to_csv(csv_outputs_dir/"df_single_no_grps_raw.csv", index=False)  
df_single_no_grps_raw.to_parquet(parquet_outputs_dir/"df_single_no_grps_raw.parquet", index=False, engine="pyarrow")

single_columns_non_numeric = [
    col for col in df_single_no_grps_raw.columns
    if col not in ["resp_id", "age", "resp_id_rand", "satisfaction"]
]
print ("Number of single non-numeric cols:", len(single_columns_non_numeric))
print("Created df_single_no_grps_raw.csv and stored in csv_outputs_dir.")
print("Single column non-numeric list defined.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Number of single cols: 24
Number of single non-numeric cols: 20
Created df_single_no_grps_raw.csv and stored in csv_outputs_dir.
Single column non-numeric list defined.
Date and Time:  2026-03-17 12:55:39


In [57]:
# 4a2. Print out to terminal all unique values
# Check unique values first except resp_id  
# Write all unique values to a single file except resp_id, age

# Optional - run again if needed
# single_columns_non_numeric = [col for col in df_single_no_grps_raw.columns if col not in ['resp_id', 'age', 'resp_id_rand', 'satisfaction']]

with open(unique_dir / "unique_single.txt", "w", encoding="utf-8") as f:
    for col in single_columns_non_numeric:
        f.write(f"{col}\n\n")
        for val in df_single_no_grps_raw[col].dropna().unique():
            f.write(f"{val}\n")
        f.write("\n")
        
print("unique_single.txt created.  Inspect unique values.")
with open(unique_dir / "unique_single.txt", "r", encoding="utf-8") as f:
    print(f.read())
print("Number of csv files in unique_dir:", len(list(unique_dir.glob("*.csv"))))
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

unique_single.txt created.  Inspect unique values.
country

Philippines
United Kingdom
Norway
Cameroon
Cambodia
Thailand
Pakistan
United Arab Emirates
Canada
Nepal
United States
Uae
Nigeria
Jordan
Egypt
Ksa
South Africa
Uganda
Congo
South Korea
Malaysia
Bangladesh
Yemen
Saudi Arabia
Peru
Perú
Kenya
Algeria
Vietnam
Ghana
India
Spain
Phillipines
Taiwan
Mexico

city_ph

Pampanga
Binan
Mandaluyong
San Quintin
Taguig
Not Applicable
Aleosan
Dasmarinas
Muntinlupa
Quezon City
Pasig
Caloocan
Davao
Paranaque
Manila
Bacolod
Makati
Batangas
Zamboanga
Roxas
Las Pinas
La Pinas
Ozamiz City
Cavite
Calamba
Binalonan
Bulacan
Imus
Cebu
Tanza
Baliwag
Lake Sebu, South Cotabato
La Union
Camarines Sur
Calbayog Samar
Cagayan De Oro
Calbayog City
Laguna
Roxas City
San Fernando
Davao Del Norte
Surigao City
San Pablo
Bukidnon
Tarlac
Rizal
Iloilo
Valencia City Bukidnon
Cabuyao
El Salvador City
Malabon City
Negros Occidental
Isabela
San Jose
Valenzuela
Pasay
Cainta
Norzagaray
Albay
Mabalacat
Antipolo
General Maria

In [10]:
# 4b. Convert unique values to starter csv files for lookup and store in unique_dir 
# → copy and edit into lookup_dir as final lookup csv 
# → unique_dir → lookup_dir after 3 types of transformations:
# → 1. direct clean 2. specific edits 3. manual edits (hitl add "clean" column) 

single_columns_non_numeric = [
    col for col in df_single_no_grps_raw.columns
    if col not in ["resp_id", "age", "resp_id_rand", "satisfaction"]
]

for col in single_columns_non_numeric:
    uniques = (
        df_single_no_grps_raw[col]
        .dropna()
        .astype(str)
        .unique()
    )
    uniques_sorted = sorted(uniques)

    out_df = pd.DataFrame({"raw": uniques_sorted})
    out_path = unique_dir / f"{col}.csv"
    out_df.to_csv(out_path, index=False, encoding="utf-8")

print("~"*80)
print("Per-column unique CSVs created in unique_dir.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Per-column unique CSVs created in unique_dir.
Date and Time:  2026-03-17 09:03:56


# 4c. Create lookup files

In [58]:
# 4c. Create temp lookup csv files (staging column) in lookup_dir using the raw column values as clean_temp
# Three types of edits: 1. no edits (direct clean), 2. specific edits, 3. manual edits (the rest)

# Direct clean (Type 1)
direct_clean = [
    "ai_study_lookup.csv", "ai_work_lookup.csv", "careerstg_lookup.csv",
    "datarole_lookup.csv", "educstat_lookup.csv", "employertype_lookup.csv",
    "gender_lookup.csv", "industry_lookup.csv", "satisfaction_lookup.csv",
    "shift_schedule_lookup.csv", "sitework_lookup.csv", "sizeteam_lookup.csv",
    "typework_lookup.csv"
]

# Specific edit files (Type 2) → go to type2 staging dir, not lookup
specific_csv_lookup_files = [
    "have_comp_degree_lookup.csv",
    "side_gig_lookup.csv",
    "salary_lookup.csv",
    "how_long_in_salary_lookup.csv",

]

# Paths
unique_dir = Path("unique_dir")
lookup_dir = Path("lookup_dir")

type1_dir = lookup_dir / "type1_dir"
type2_dir = lookup_dir / "type2_dir"
type3_dir = lookup_dir / "type3_dir"

# Ensure directories exist
type1_dir.mkdir(parents=True, exist_ok=True)
type2_dir.mkdir(parents=True, exist_ok=True)
type3_dir.mkdir(parents=True, exist_ok=True)

for file in unique_dir.glob("*.csv"):
    df = pd.read_csv(file, dtype=str)

    # Always create clean_temp as staging column
    df["clean_temp"] = df["raw"]

    lookup_filename = f"{file.stem}_lookup.csv"

    # Type 1: direct clean
    if lookup_filename in direct_clean:
        df = df.rename(columns={"clean_temp": "clean"})
        out_path1 = type1_dir / lookup_filename
        df.to_csv(out_path1, index=False, encoding="utf-8")
        print(f"Type 1 lookup created: {out_path1}")
        continue

    # Type 2: specific edits → staging file, not lookup
    if lookup_filename in specific_csv_lookup_files:
        staging_filename_type2 = f"{file.stem}_staging.csv"
        out_path2 = type2_dir / staging_filename_type2
        df.to_csv(out_path2, index=False, encoding="utf-8")
        print(f"Type 2 staging file created: {out_path2}")
        continue

    # All the rest - manual edits
    # Type 3: manual edits → normal lookup with clean_temp
    staging_filename_type3 = f"{file.stem}_staging.csv"
    out_path3 = type3_dir / staging_filename_type3
    df.to_csv(out_path3, index=False, encoding="utf-8")
    print(f"Type 3 lookup created: {out_path3}")

print("All lookup and staging files generated.")
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Type 1 lookup created: lookup_dir\type1_dir\ai_study_lookup.csv
Type 1 lookup created: lookup_dir\type1_dir\ai_work_lookup.csv
Type 1 lookup created: lookup_dir\type1_dir\careerstg_lookup.csv
Type 3 lookup created: lookup_dir\type3_dir\city_non_ph_staging.csv
Type 3 lookup created: lookup_dir\type3_dir\city_ph_staging.csv
Type 3 lookup created: lookup_dir\type3_dir\comp_related_degree_staging.csv
Type 3 lookup created: lookup_dir\type3_dir\country_staging.csv
Type 1 lookup created: lookup_dir\type1_dir\datarole_lookup.csv
Type 1 lookup created: lookup_dir\type1_dir\educstat_lookup.csv
Type 1 lookup created: lookup_dir\type1_dir\employertype_lookup.csv
Type 1 lookup created: lookup_dir\type1_dir\gender_lookup.csv
Type 2 staging file created: lookup_dir\type2_dir\have_comp_degree_staging.csv
Type 2 staging file created: lookup_dir\type2_dir\how_long_in_salary_staging.csv
Type 1 lookup created: lookup_dir\type1_dir\industry_lookup.csv
Type 2 staging file created: lookup_dir\type2_dir\sala

In [59]:
# 4d. Special lookup files given different wording vs. labels for charts
# have_comp_degree, side_gig, salary, how_long_in_salary
# Create map → apply map function & drop temp → export


# Define all maps → feeds into dictionary → replace "clean_temp" in lookup
map_comp_degree = {
    "No computer-related or data-related degree": "Not have",
    "Yes, I have a computer-related or data-related degree": "Have comp or data degree"
}

map_side_gig = {
    "I HAD at least one side gig in the past 12 months": "Side gig p12m",
    "I did NOT have side gigs in the past 12 months": "No side gig p12m"
}

map_salary = {
    "100,001 to 125,000":"100k+ to 125k",
    "125,001 to 250,000":"125k+ to 250k",
    "15,000 and below":"15k and below",
    "15,001 to 25,000":"15k+ to 25k",
    "25,001 to 35,000":"25k+ to 35k",
    "250,001 and above":"250k and above",
    "35,001 to 45,000":"35k+ to 45k",
    "45,001 to 55,000":"45k+ to 55k",
    "55,001 to 65,000":"55k+ to 65k",
    "65,001 to 75,000":"65k+ to 75k",
    "75,001 to 85,000":"75k+ to 85k",
    "85,001 to 95,000":"85k+ to 95k",
    "95,001 to 100,000":"95k+ to 100k"
}

map_how_long_in_salary = {
    "3 months or less": "a. 3mos or less",
    "More than 3 to 6 months": "b. >3 to 6mos",
    "More than 6 months less than 1 year": "c. >6mos to 1yr",
    "More than 1 year to 2 years": "d. >1yr to 2yrs",
    "More than 2 years to 5 years": "e. >2 yrs to 5yrs",
    "More than 5 years to 10 years": "f. >5 yrs to 10yrs",
    "More than 10 years": "g. >10yrs",
    "None of the above": "h. Not Applicable",
    "Not Applicable": "Not Applicable"
}

# Dictionary for all special mappings 
all_mappings = {
    "have_comp_degree": map_comp_degree,
    "side_gig": map_side_gig,
    "salary": map_salary,
    "how_long_in_salary": map_how_long_in_salary
}

# Function apply_special_map → lookup files
def apply_special_map(col_name, mapping_dict, lookup_dir):
    """
    Reads, maps, cleans temp lookup csvs and and exports lookup CSVs.
    Uses 'clean_temp' as the common source column.
    """
    file_path2 =  type2_dir / f"{col_name}_staging.csv"
    output_path2 = type2_dir/ f"{col_name}_lookup.csv"
    df = pd.read_csv(file_path2, dtype=str)    
    df["clean"] = df["clean_temp"].map(mapping_dict).fillna(df["clean_temp"])
    
    # Remove the staging column and overwrite the source file
    df.drop(columns=["clean_temp"], inplace=True)
    df.to_csv(output_path2, index=False)

# Apply function to special columns
for col, mapping in all_mappings.items():
    apply_special_map(col, mapping, lookup_dir)

print("Special edits type2 lookup files created.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Special edits type2 lookup files created.
Date and Time:  2026-03-17 12:58:15


In [60]:
# 4d. Check files to manually edit → Type 3 edit

# This is unavoidable. Some columns really need human input.  Attempt to use AI resulted in missed important items (ex. BS Business Administration ← failed to convert to  BSBA Major Unspecified, Two year computer course ← failed to convert to Associate, and more)

# Initialize {set} object to contain all file names in type3_dir
manual_edit_files= {f"{file.stem}.csv" for file in type3_dir.glob("*.csv")}

print("Lookup files to manually edit:")
print(manual_edit_files)
print("~"*50)
print("REMOVE ALL COMMAS AND NON LETTER CHARACTERS, OPTIONS THAT CAN BE CONVERGED TO ONE, WRONG GRAMMAR")
print("AND THEN COPY AND PASTE INTO THE CORRECT FILENAME IN THE MANUAL_EDIT FOLDER")
print("~"*50)
print("COPY ALL THE FINAL LOOKUP FILES TO ROOT LOOKUP_DIR")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Lookup files to manually edit:
{'city_ph_staging.csv', 'comp_related_degree_staging.csv', 'city_non_ph_staging.csv', 'country_staging.csv', 'city_non_ph_lookup.csv', 'city_ph_lookup.csv', 'comp_related_degree_lookup.csv', 'country_lookup.csv'}
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
REMOVE ALL COMMAS AND NON LETTER CHARACTERS, OPTIONS THAT CAN BE CONVERGED TO ONE, WRONG GRAMMAR
AND THEN COPY AND PASTE INTO THE CORRECT FILENAME IN THE MANUAL_EDIT FOLDER
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
COPY ALL THE FINAL LOOKUP FILES TO ROOT LOOKUP_DIR
Date and Time:  2026-03-17 12:58:31


## REMINDER: COPY AND PASTE MANUAL EDIT LOOKUP FILES IN A MANUAL_EDIT_IMPT SUBFOLDER NOW.

In [61]:
# 4e.  Copy all the final lookup files to root lookup_dir

import shutil

# List of source directories
source_dirs = [type1_dir, type2_dir, type3_dir]

# Copy all *_lookup.csv files into lookup_dir
for src in source_dirs:
    for file in src.glob("*_lookup.csv"):
        dest = lookup_dir / file.name
        shutil.copy(file, dest)
        print(f"Copied: {file} -> {dest}")

print("All lookup files copied into lookup_dir.")
print("Number of lookup files copied to lookup_dir:", len(list(lookup_dir.glob("*.csv"))))
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Copied: lookup_dir\type1_dir\ai_study_lookup.csv -> lookup_dir\ai_study_lookup.csv
Copied: lookup_dir\type1_dir\ai_work_lookup.csv -> lookup_dir\ai_work_lookup.csv
Copied: lookup_dir\type1_dir\careerstg_lookup.csv -> lookup_dir\careerstg_lookup.csv
Copied: lookup_dir\type1_dir\datarole_lookup.csv -> lookup_dir\datarole_lookup.csv
Copied: lookup_dir\type1_dir\educstat_lookup.csv -> lookup_dir\educstat_lookup.csv
Copied: lookup_dir\type1_dir\employertype_lookup.csv -> lookup_dir\employertype_lookup.csv
Copied: lookup_dir\type1_dir\gender_lookup.csv -> lookup_dir\gender_lookup.csv
Copied: lookup_dir\type1_dir\industry_lookup.csv -> lookup_dir\industry_lookup.csv
Copied: lookup_dir\type1_dir\shift_schedule_lookup.csv -> lookup_dir\shift_schedule_lookup.csv
Copied: lookup_dir\type1_dir\sitework_lookup.csv -> lookup_dir\sitework_lookup.csv
Copied: lookup_dir\type1_dir\sizeteam_lookup.csv -> lookup_dir\sizeteam_lookup.csv
Copied: lookup_dir\type1_dir\typework_lookup.csv -> lookup_dir\typework

In [62]:
# 4f. Add Not Applicable to lookup files, both raw and clean columns if not yet present

# Define the directory containing your data files
lookup_dir = Path("lookup_dir")

for file in lookup_dir.glob("*.csv"):
    # Load the data into a structured table format
    # We use 'latin-1' to ensure compatibility with various data sources
    df = pd.read_csv(file, dtype=str, keep_default_na=False, encoding="latin-1")

    # Verify that the necessary 'raw' and 'clean' categories exist
    for col in ("raw", "clean"):
        if col not in df.columns:
            df[col] = ""

    # Check if the specific 'Not Applicable' pair already exists in the data
    na_exists = ((df["raw"] == "Not Applicable") & (df["clean"] == "Not Applicable")).any()

    # Append the row only if it is missing from the current records
    if not na_exists:
        df.loc[len(df)] = {"raw": "Not Applicable", "clean": "Not Applicable"}

        # Overwrite the file with the updated information in a standardized format
        df.to_csv(file, index=False, encoding="utf-8")
        print(f"Successfully updated: {file.name}")

print ("Not Applicable successfully added to lookups if not yet present. ")
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Successfully updated: ai_study_lookup.csv
Successfully updated: ai_work_lookup.csv
Successfully updated: careerstg_lookup.csv
Successfully updated: country_lookup.csv
Successfully updated: educstat_lookup.csv
Successfully updated: gender_lookup.csv
Successfully updated: have_comp_degree_lookup.csv
Not Applicable successfully added to lookups if not yet present. 
Date and Time:  2026-03-17 13:02:10


In [ ]:
# 5. Define columns for lookups (Single-Respons Non Numeric Columns)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# At this point, lookups are already created in lookup_dir and manually revised.
# Single columns in df_raw will be transformed using lookups. 
# A new df_single_no_grps_raw, the base df for lookups, has been created.
# df_single_no_grps_raw -> df_single_no_grps (lookups applied) 

# Optional 
# Re-run the single_column_no_grps_raw definition in 4a (Optional)
# Re-run the df_single_no_grps_raw storage to csv_outputs_dir(Optional)
# Re-run the single_column_non_numerics definition

# 5. Apply Lookups

In [63]:
# Very specific edit not captured by global cleaning.
df_single_no_grps_raw['country'] = df_single_no_grps_raw['country'].str.replace('Perú', 'Peru', regex=False)

In [ ]:
# 5a. Create apply_lookup function
# Apply lookup function to single - response non-numeric columns in df_single_no_grps_raw
# Unique values → apply_lookup function → convert df_single_no_grps.csv using lookup
# Print unmatched items and print confirmed transformations per column


# Create function
def apply_lookup(df, column):

    # Check if lookup file for column exists
    lookup_file = lookup_dir / f"{column}_lookup.csv"
    if not lookup_file.exists():
        print(f"{column} lookup file not found.")
        # Import df_single_no_grps_raw as df
        return df

    # Import the lookup file as map_df
    map_df = pd.read_csv(lookup_file, encoding="windows-1252", quotechar='"', quoting=csv.QUOTE_MINIMAL, dtype=str, keep_default_na=False)

    # Clean raw and clean columns in map_df(lookup file)
    map_df["raw"] = map_df["raw"].fillna("")
    map_df["clean"] = map_df["clean"].fillna("")

    # List unique values in the columm from df_single_no_grps_raw
    list_raw = list(df[column].unique())
    # List unique values in the raw column of the lookup file
    list_lookup = list(map_df["raw"].unique())

    # Loop through unique values in df_single_no_grps_raw
    # If not in the lookup raw column, add to unmatched list
    unmatched = [x for x in list_raw if x not in list_lookup]
    if not unmatched:
        print (f"All items in {column} of raw file are matched.")
    else:
        print(f"Unmatched items in raw {column}:")
        for item in unmatched:
            print(repr(item))  # official string representation

    mapping = dict(zip(map_df["raw"], map_df["clean"]))

    df[column] = df[column].map(mapping).fillna("Not Applicable")
    print (f"Mapping done for {column}")
    return df

# Recall the single_columns_non_numeric list was created in 4a.
for col in single_columns_non_numeric:
    df_single_no_grps= apply_lookup(df_single_no_grps_raw, col)
    print("~"*50)

# Logging...

print('Run date: ', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
# Create df_single_no_grps csv file after transforming raw file
df_single_no_grps.to_csv(csv_outputs_dir/ "df_single_no_grps.csv", index = False)
df_single_no_grps.to_parquet(parquet_outputs_dir/"df_single_no_grps.parquet", index=False, engine="pyarrow")

All items in country of raw file are matched.
Mapping done for country
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
All items in city_ph of raw file are matched.
Mapping done for city_ph
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
All items in city_non_ph of raw file are matched.
Mapping done for city_non_ph
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
All items in gender of raw file are matched.
Mapping done for gender
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
All items in educstat of raw file are matched.
Mapping done for educstat
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
All items in have_comp_degree of raw file are matched.
Mapping done for have_comp_degree
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
All items in comp_related_degree of raw file are matched.
Mapping done for comp_related_degree
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
All items in careerstg of raw file are matched.
Mapping done for careerstg
~~~~~~~~~~~~~~~~~~~~~

# 5aa HITL Inspect unmatched items.  Manually edit lookup files if needed.
## LLM does not know context yet, only does fuzzy matches as of this update.
- No need to run lookup again - second pre-cleaning.
- At this point, df_single_no_grps values are already updated, so cannot re-run item 5. Lookups. 
- As of this update, there was no need to run 5a onwards. All items matched, and minimal edits in VSCode.
- If really need to run the subset:
    - Just edit the lookups manually in VSCode, not Excel.
    - Or define the subset, noting that artificial unmatched items will appear.

In [ ]:
# 5b. Re-run lookups only for subset based on the output above to minimize artificial mismatch items. (Optional)
# Do not re-run cell #5a because some df_raw columns are already standardized and replaced.
# So some 'unmatched' items are artificial because the df_single_no_grps raw column is already transformed to clean column.  
# This list of subset columns to re-run was the result of the first pre-cleaning.
# Second pre-cleaning did not have any unmatched items anymore.

subset_rerun = ['city_non_ph', 'comp_related_degree', 'employertype', 'industry', 'salary', 'how_long_in_salary', 'typework', 'sitework', 'shift_schedule', 'datarole', 'side_gig', 'satisfaction', 'sizeteam']

df_single_no_grps[subset_rerun] = df_single_no_grps[subset_rerun].fillna("Not Applicable")

for col in subset_rerun:
    df_single_no_grps= apply_lookup(df_single_no_grps, col)
    print(f"\n{col} lookup done.\n")
    print("~"*20)

print("Proceed to inspection again, next cell.")


# 5c. EDA boilerplate

In [66]:
# 5c. Inspect the updated df_single_no_grps unique values and export compilation as csv file(Optional)
# During pre-cleaning - this was only to check if unforeseen unmatched values still appeared.

csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)
df_single_no_grps = pd.read_csv(csv_outputs_dir/"df_single_no_grps.csv")
single_cols_no_grps = [
    "resp_id", "country", "city_ph", "city_non_ph", "gender", "age", "educstat", "have_comp_degree", "comp_related_degree", "careerstg", "employertype", "industry", "salary", "how_long_in_salary", "typework", "sitework", "shift_schedule", "datarole", "side_gig", "satisfaction", "sizeteam", "ai_work", "ai_study", "resp_id_rand"
]

parsed_rows = []  # <-- define this once before the loop, saved using pd.DataFrame

def explore_categorical_cols_to_csv(col, df):
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')
    print(f'Exploration of {col} column')
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')

    # Header row for CSV
    parsed_rows.append({
        "column": col,
        "label": f"Description of {col} ~~~~~~~~~~",
        "value": ""
    })

    unique_count = df[col].nunique()

    if unique_count > 100:
        msg = f"Skipping {col} (unique values = {unique_count}, exceeds threshold of 100)."
        print(msg)
        parsed_rows.append({
            "column": col,
            "label": "skip_reason",
            "value": msg
        })
        print()
        return

    print(f'Description of {col} ~~~~~~~~~~')
    desc = df[col].describe()
    print(desc)
    print()

    # Parsed describe() rows
    for label, value in desc.items():
        parsed_rows.append({
            "column": col,
            "label": label,
            "value": value
        })

    print(f'Value counts of {col} ~~~~~~~~~~')
    vc = df[col].value_counts(dropna=False)
    print(vc)
    print()

    # Parsed value_counts() rows
    for label, value in vc.items():
        parsed_rows.append({
            "column": col,
            "label": label,
            "value": value
        })

# Apply eda function 
for col in single_cols_no_grps:
    explore_categorical_cols_to_csv(col, df_single_no_grps)

summary_df_single_no_grps = pd.DataFrame(parsed_rows)
summary_df_single_no_grps.to_csv(os.path.join(csv_outputs_dir, "summary_single_no_grps.csv"), index=False)


print ("At this point, all unique items should be clean.")
print ("Still no age_grp because we need exact age for the duplication check.")
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Exploration of resp_id column
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Skipping resp_id (unique values = 1861, exceeds threshold of 100).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Exploration of country column
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Description of country ~~~~~~~~~~
count            1861
unique             31
top       Philippines
freq             1798
Name: country, dtype: object

Value counts of country ~~~~~~~~~~
country
Philippines       1798
Nigeria             11
Egypt                7
UAE                  5
Canada               3
United Kingdom       3
KSA                  3
United States        2
Nepal                2
Uganda               2
Ghana                2
Kenya                2
Peru                 2
South Africa         2
Pakistan             1
Jordan               1
Cambodia             1
Thailand             1
Norway               1
Cameroon             1
Malaysia             

# 5d. Check similarity - single response columns

In [67]:
# 5d. Check similarity - preliminary duplicate checks among single-response columns first
# Set up the cleaned single‑response dataframe that we will compare
df_sim_raw = df_single_no_grps.copy()

# List the columns we will use to score similarity
compare_cols = [
    "country", "city_ph", "city_non_ph", "gender", "age", "educstat",
    "have_comp_degree", "comp_related_degree", "careerstg", "employertype",
    "industry", "salary", "how_long_in_salary", "typework", "sitework",
    "shift_schedule", "datarole", "side_gig", "satisfaction", "sizeteam",
    "ai_work", "ai_study"
]

# Initialize list to hold the similarity results for each respondent
results = []

# Loop through each respondent one at a time
for idx, row in df_sim_raw.iterrows():

    # Create a row‑vs‑all‑rows comparison so we can see who looks similar
    # Compare is the boolean dataframe that contains results True/False per column
    compare = df_sim_raw[compare_cols] == row[compare_cols]

    # Count how many fields match for each respondent
    match_counts = compare.sum(axis=1)

    # Remove the self‑match so it doesn't interfere with scoring
    match_counts.loc[idx] = -1

    # Get the highest match score for this respondent
    max_score = match_counts.max()

    # Grab all respondents who hit that same top score
    best_idxs = match_counts[match_counts == max_score].index.tolist()

    # Save the result so we can review or export later
    results.append({
        "resp_id": row["resp_id"],
        "best_match_resp_ids": df_sim_raw.loc[best_idxs, "resp_id"].tolist(),
        "best_score": int(max_score),
        "total": len(compare_cols),
        "similarity": max_score / len(compare_cols)
    })

# Turn the collected results into a dataframe
df_sim = pd.DataFrame(results)
df_sim


,resp_id,best_match_resp_ids,best_score,total,similarity
0,1,[1373],15,22,0.681818
1,2,"[232, 420, 1413, 1445]",14,22,0.636364
2,3,[402],18,22,0.818182
3,4,[222],15,22,0.681818
4,5,"[123, 125, 163, 385, 454, 1168]",14,22,0.636364
...,...,...,...,...,...
1856,1857,[1303],19,22,0.863636
1857,1858,"[585, 651, 697, 776, 922, 1080, 1144, 1380, 15...",18,22,0.818182
1858,1859,[656],14,22,0.636364
1859,1860,[858],19,22,0.863636


In [68]:
# 5e. Inspect the highest scores
df_sim_sorted = df_sim.sort_values(by = 'similarity', ascending = False)
print(df_sim_sorted[['resp_id', 'best_match_resp_ids', 'similarity']].head(50))
df_sim_sorted.to_csv(csv_outputs_dir/ "df_sim_sorted.csv", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

      resp_id                                best_match_resp_ids  similarity
470       471                                              [502]    1.000000
1337     1338                                             [1190]    1.000000
1122     1123                                             [1628]    1.000000
1188     1189                                              [195]    1.000000
1189     1190                                             [1338]    1.000000
1417     1418                                              [239]    1.000000
1418     1419                                              [961]    1.000000
1210     1211                                              [845]    1.000000
501       502                                              [471]    1.000000
1284     1285                                              [839]    1.000000
1290     1291                                             [1831]    1.000000
217       218                                             [1315]    1.000000

In [74]:
# 5f. Filter respondents most likely to be duplicates 
# Score == 1 & only one best_match_resp_ids using apply(len) == 1)
df_sim_sorted = df_sim.sort_values(by = 'similarity', ascending = False)
df_likely_dups = df_sim_sorted[
    (df_sim_sorted['best_match_resp_ids'].apply(len) == 1) &
    (df_sim_sorted['similarity'] == 1)
]

print(df_likely_dups.head(100))
print("Number of pairs of resp_ids who likely have duplicates: ", len(df_likely_dups)/2)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Export the df_likely_dups to csv outputs
df_likely_dups.to_csv(csv_outputs_dir/"table_of_likely_dups.csv", index = False)


      resp_id best_match_resp_ids  best_score  total  similarity
470       471               [502]          22     22         1.0
1337     1338              [1190]          22     22         1.0
1122     1123              [1628]          22     22         1.0
1188     1189               [195]          22     22         1.0
1189     1190              [1338]          22     22         1.0
1417     1418               [239]          22     22         1.0
1418     1419               [961]          22     22         1.0
1210     1211               [845]          22     22         1.0
501       502               [471]          22     22         1.0
1284     1285               [839]          22     22         1.0
1290     1291              [1831]          22     22         1.0
217       218              [1315]          22     22         1.0
201       202               [781]          22     22         1.0
1533     1534              [1624]          22     22         1.0
1119     1120            

In [78]:
# 5f. Create df possible duplicates based on the single response columns.

df_possible_duplicates = df_single_no_grps.merge(df_likely_dups, how = "inner", on = "resp_id")
df_possible_duplicates.to_csv(csv_outputs_dir/ "df_possible_duplicates.csv", index = False)
df_possible_duplicates

,resp_id,country,city_ph,city_non_ph,gender,age,educstat,have_comp_degree,comp_related_degree,careerstg,...,side_gig,satisfaction,sizeteam,ai_work,ai_study,resp_id_rand,best_match_resp_ids,best_score,total,similarity
0,101,Philippines,Isabela,Not Applicable,Male,21,Bachelor's or College Degree,Have comp or data degree,BS Information Technology,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-303c21,[357],22,22,1.0
1,195,Philippines,Quezon City,Not Applicable,Male,22,Bachelor's or College Degree,Not have,Not Applicable,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-b06d77,[1189],22,22,1.0
2,198,Philippines,Rizal,Not Applicable,Female,23,Bachelor's or College Degree,Have comp or data degree,BS Information Technology,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-3cf0b0,[1337],22,22,1.0
3,202,Philippines,Caloocan,Not Applicable,Male,22,Bachelor's or College Degree,Have comp or data degree,BS Computer Science,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-3857f0,[781],22,22,1.0
4,218,Philippines,Caloocan,Not Applicable,Male,18,Bachelor's or College Degree,Have comp or data degree,BS Information Technology,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-e5751e,[1315],22,22,1.0
5,239,Philippines,Zambales,Not Applicable,Female,23,Bachelor's or College Degree,Have comp or data degree,BS Computer Engineering,Looking for data-related work - actively seeki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-1f27c8,[1418],22,22,1.0
6,288,Philippines,Calamba,Not Applicable,Male,21,Bachelor's or College Degree,Have comp or data degree,BS Computer Science,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-1a4dbb,[856],22,22,1.0
7,306,Philippines,Quezon City,Not Applicable,Male,21,High school graduate,Have comp or data degree,BS Applied Statistics,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"No, and I don't plan to.","Yes, I use AI tools MONTHLY or LESS frequently...",2025-0b32fb,[1031],22,22,1.0
8,337,Philippines,Cavite,Not Applicable,Male,22,Bachelor's or College Degree,Have comp or data degree,BS Computer Science,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-f12e43,[577],22,22,1.0
9,343,Philippines,Batangas,Not Applicable,Male,22,Bachelor's or College Degree,Have comp or data degree,BS Computer Science,Student - currently studying and not yet worki...,...,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-888e8e,[1382],22,22,1.0


# 6. Create groups for age, salary and ph regions
## This comes after similarity scoring.  Groups are customized values.

In [84]:
# 6a. Create age_grp in df_single_no_grps

# Initialize the df_single_with_grps (Optional)
#df_single_no_grps = pd.read_csv("csv_outputs_dir/df_single_no_grps_after_lookup.csv")

df_single_with_grps_age = df_single_no_grps.copy()
df_single_with_grps_age = df_single_with_grps_age[['resp_id', 'resp_id_rand', 'age']]

# Bin the 'age' variable into 'age_grp'
bins = [-np.inf, 19, 25, 30, 35, 40, 45, 50, 55, np.inf]
labels = ["<19", "20 to 24", "25 to 29", "30 to 34", "35 to 39", "40 to 44", "45 to 49", "50 to 54", "55+"]
df_single_with_grps_age['age_grp'] = pd.cut(df_single_no_grps['age'], bins=bins, labels=labels, right=False, include_lowest=True)
df_single_with_grps_age.to_csv(csv_outputs_dir/ "df_single_with_grps_age.csv", index = False)
print("age_grp column added to df_single_with_grps_age.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


age_grp column added to df_single_with_grps_age.
Date and Time:  2026-03-17 13:50:23


In [82]:
# Check df_single_with_grps
df_single_with_grps_age

,resp_id,resp_id_rand,age,age_grp
0,1,2025-eed25f,23,20 to 24
1,2,2025-c107bd,36,35 to 39
2,3,2025-b053b8,32,30 to 34
3,4,2025-a03238,34,30 to 34
4,5,2025-ae4e76,29,25 to 29
...,...,...,...,...
1856,1857,2025-05787a,27,25 to 29
1857,1858,2025-e0860f,28,25 to 29
1858,1859,2025-0b73c1,35,35 to 39
1859,1860,2025-571482,23,20 to 24


In [85]:
# 6b. Create column salary_broader for plotly splits

# Define broader salary bands using match-case
def categorize_salary(salary):
    match salary:
        case x if x in ['15k and below', '15k+ to 25k', '25k+ to 35k']:
            return "35k and less"
        case x if x in ['35k+ to 45k', '45k+ to 55k', '55k+ to 65k', '65k+ to 75k']:
            return "35k+ to 75k"
        case x if x in ['75k+ to 85k', '85k+ to 95k', '95k+ to 100k']:
            return "75k+ to 100k"
        case x if x in ['100k+ to 125k', '125k+ to 250k', '250k and above']:
            return "100k and above"   
        case x if x == 'Not Applicable':
            return "Not Applicable"
        case _:
            return "Unknown"
df_single_with_grps_salary = df_single_no_grps.copy()
df_single_with_grps_salary = df_single_with_grps_salary[['resp_id', 'resp_id_rand', 'salary']]        
df_single_with_grps_salary['salary_broader'] = df_single_with_grps_salary['salary'].apply(categorize_salary)
df_single_with_grps_salary.to_csv("csv_outputs_dir/df_single_with_grps_salary.csv", index = False )

print("Added salary_broader column to df_single_with_grps_salary. ")
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Added salary_broader column to df_single_with_grps_salary. 
Date and Time:  2026-03-17 13:51:34


In [87]:
# Check if any salary_broader == 'Unknown'
unknown_salary_check = df_single_with_grps_salary[df_single_with_grps_salary['salary_broader'] == "Unknown"][['resp_id', 'salary', 'salary_broader']]
unknown_salary_check

,resp_id,salary,salary_broader


# 6c. AI Integration: Calling Gemini API # HITL for Region Group
- Need human intervention as Gemini has a lot of "Unknown" classifications
- Use the google ai studio app for initial ph region lookups


In [ ]:
# 6c. Initial lookup file created as city_ph_lookup_categorized.csv
# categorize_locations.py
# Prework: # pip install pandas google-genai pydantic
# API key created for "Gemini Project" stored in the secrets folder as api_key.txt

"""
Teaching the Gemini AI how to act as a data entry clerk for survey results. By using Pydantic BaseModels, we are giving the AI a "form" to fill out so it doesn't just send back a messy paragraph of text, but a structured list that our code can easily understand.

"""
import os
import json
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# Load the API key
key_path = os.path.join("secrets", "api_key.txt")

try:
    with open(key_path, "r") as file:
        # .strip() to remove any hidden newlines or spaces from the text file
        gemini_api_key = file.read().strip() 
except FileNotFoundError:
    print(f" Error: Could not find the API key file at {key_path}")
    # Stop execution if the key isn't found
    raise 

# Initialize the Gemini client
client = genai.Client(api_key=gemini_api_key)

# Define the input and output schemas
class CategorizedLocation(BaseModel):
    original: str = Field(description="The original location string provided")
    region: str = Field(description="The categorized region: 'Metro Manila', 'Balance Luzon', 'Visayas', 'Mindanao', or 'Unknown'")

class LocationBatchResponse(BaseModel):
    results: list[CategorizedLocation]

def categorize_locations(csv_path: str, column_name: str, output_path: str):
    print(f"\nReading {csv_path}...")
    df = pd.read_csv(csv_path)

    if column_name not in df.columns:
        print(f"Error: Column '{column_name}' not found in the CSV.")
        return

    # Extract unique locations
    unique_locations = df[column_name].dropna().astype(str).str.strip().unique().tolist()
    unique_locations = [loc for loc in unique_locations if loc]

    if not unique_locations:
        print("No valid locations found.")
        return

    print(f"Found {len(unique_locations)} unique locations. Categorizing...")

    location_map = {}
    batch_size = 50 

    for i in range(0, len(unique_locations), batch_size):
        batch = unique_locations[i:i + batch_size]
        print(f"Processing batch {i//batch_size + 1} of {(len(unique_locations) - 1)//batch_size + 1}...")

        prompt = f"""
        Categorize the following locations in the Philippines into one of these exactly 4 regions:
        "Metro Manila", "Balance Luzon", "Visayas", "Mindanao".
        If the location cannot be determined or is outside the Philippines, categorize it as "Unknown".
        
        Locations to categorize:
        {batch}
        """

        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=LocationBatchResponse,
                    temperature=0.1,
                ),
            )

            data = json.loads(response.text)
            for item in data.get("results", []):
                location_map[item["original"]] = item["region"]
                
        except Exception as e:
            print(f"Error processing batch: {e}")

    # Map back to the dataframe
    print("\nMapping results back to the dataset...")
    df['Region'] = df[column_name].astype(str).str.strip().map(location_map).fillna('Unknown')

    df.to_csv(output_path, index=False)
    print(f"Successfully saved categorized data to {output_path}")

# --- Run the function (no client.close() needed)---
input_file = "lookup_dir/city_ph_lookup.csv"
location_column = "clean" 
output_file = "lookup_dir/city_ph_lookup_categorized.csv"

categorize_locations(input_file, location_column, output_file)
print ("Latest run of categorized locations: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Reading lookup_dir/city_ph_lookup.csv...
Found 164 unique locations. Categorizing...
Processing batch 1 of 4...
Processing batch 2 of 4...
Processing batch 3 of 4...
Processing batch 4 of 4...

Mapping results back to the dataset...
Successfully saved categorized data to lookup_dir/city_ph_lookup_categorized.csv
Latest run of categorized locations:  2026-03-17 14:33:22


In [103]:
# 6d. Check city ph groups Unknowns
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
df_city_ph_lookup_categorized = pd.read_csv("lookup_dir/city_ph_lookup_categorized.csv")
df_city_ph_lookup_categorized[df_city_ph_lookup_categorized["Region"] == "Unknown"]

Date and Time:  2026-03-17 14:34:17


,raw,clean,Region
0,Accra,Not Applicable,Unknown
4,Alexanderia,Not Applicable,Unknown
5,Alexandria,Not Applicable,Unknown
6,Amman,Not Applicable,Unknown
40,Cairo,Not Applicable,Unknown
49,Cape Town,Not Applicable,Unknown
64,Diplo,Not Applicable,Unknown
66,Dolores,Dolores,Unknown
81,Isabela,Isabela,Unknown
86,Kinshasa,Not Applicable,Unknown


In [106]:
# 6e. Replace all Unknowns with "Not Applicable"
df_city_ph_lookup_grps_staging = df_city_ph_lookup_categorized.copy()
df_city_ph_lookup_grps_staging["Region"] = df_city_ph_lookup_grps_staging["Region"].replace("Unknown", "Not Applicable")

# Export to csv
df_city_ph_lookup_grps_staging.to_csv("lookup_dir/city_ph_lookup_grps_staging.csv", index = False)

# Check
print(df_city_ph_lookup_grps_staging[df_city_ph_lookup_grps_staging["Region"] == "Unknown"])
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Empty DataFrame
Columns: [raw, clean, Region]
Index: []
Date and Time:  2026-03-17 14:45:54


In [119]:
# check 
df_city_ph_lookup_grps = df_city_ph_lookup_grps_staging.copy()
df_city_ph_lookup_grps =df_city_ph_lookup_grps.rename(columns={"clean": "city_ph", "Region": "region_ph"})
df_city_ph_lookup_grps =df_city_ph_lookup_grps.drop(columns=["raw"]).drop_duplicates(subset=None, keep="first", inplace=False, ignore_index=False)
df_city_ph_lookup_grps.to_csv(lookup_dir/ "city_ph_lookup_grps.csv", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
df_city_ph_lookup_grps

Date and Time:  2026-03-17 15:01:02


,city_ph,region_ph
0,Not Applicable,Not Applicable
1,Aklan,Visayas
2,Albay,Balance Luzon
3,Aleosan,Mindanao
7,Angeles,Balance Luzon
...,...,...
183,Valenzuela,Metro Manila
184,Virac,Balance Luzon
185,Western Samar,Visayas
186,Zambales,Balance Luzon


### Recall we decided not to edit the city_ph responses of country != 'Philippines
- This is where we filter. 
- city_ph cleaned here (post-similarity scoring) intentionally.
- Duplicate detection in Stage 5 runs on raw values to avoid cleaning-induced score inflation.
- Raw city_ph values → similarity scoring → flag duplicates → THEN clean city_ph
- Clean city_ph → similarity scoring → flag duplicates  (false positive duplicate risk)

In [ ]:
# 6g. Add the region column to df_single_no_grps - ONLY for Philippines respondents
# Create a mask for Philippines respondents
ph_mask = df_single_no_grps['country'] == 'Philippines'

# Merge region_ph only for Philippines respondents
df_single_with_grps_region = df_single_no_grps.copy()

# Left merge only for Philippines respondents, others get NaN initially
df_single_with_grps_region = df_single_with_grps_region.merge(
    df_city_ph_lookup_grps[["city_ph", "region_ph"]], 
    how="left", 
    on="city_ph"
)

# Assign 'Not Applicable' for non-Philippines respondents and missing regions
df_single_with_grps_region['region_ph'] = df_single_with_grps_region['region_ph'].fillna('Not Applicable')

# For non-Philippines respondents, ensure they get 'Not Applicable' (override any merge result)
df_single_with_grps_region.loc[~ph_mask, 'region_ph'] = 'Not Applicable'

# Select final columns
df_single_with_grps_region = df_single_with_grps_region[["resp_id", "region_ph"]]

# Save
df_single_with_grps_region.to_csv(csv_outputs_dir/"df_single_with_grps_region.csv", index=False)

# Check
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
df_single_with_grps_region


In [ ]:
# 6h. Merge groups to df_single_no_grps
# Ensure only two columns are present in the dfs to be merged

df_single_with_grps_age_merge = df_single_with_grps_age[["resp_id", "age_grp"]]
df_single_with_grps_salary_merge= df_single_with_grps_salary[["resp_id", "salary_broader"]]
df_single_with_grps_region_merge = df_single_with_grps_region[["resp_id", "region_ph"]]

df_single_with_grps = df_single_no_grps.merge(
    df_single_with_grps_region_merge, 
    how="left", 
    on="resp_id"
).merge(
    df_single_with_grps_age_merge, 
    how="left", 
    on="resp_id"
).merge(
    df_single_with_grps_salary_merge,
    how="left",
    on="resp_id"
)

df_single_with_grps.to_csv(csv_outputs_dir/"df_single_with_grps.csv", index = False)
df_single_with_grps.to_parquet(parquet_outputs_dir/"df_single_with_grps.parquet", index = False, engine="pyarrow")
print ("Created df_single_with_grps.csv and df_single_with_grps.parquet")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Created df_single_with_grps.csv and df_single_with_grps.parquet
Date and Time:  2026-03-17 15:59:34


In [ ]:
# 6i. Check df_single_with_grps
df_single_with_grps = pd.read_parquet(parquet_outputs_dir/"df_single_with_grps.parquet")

df_single_with_grps

,resp_id,country,city_ph,city_non_ph,gender,age,educstat,have_comp_degree,comp_related_degree,careerstg,...,datarole,side_gig,satisfaction,sizeteam,ai_work,ai_study,resp_id_rand,region_ph,age_grp,salary_broader
0,1,Philippines,Pampanga,Not Applicable,Male,23,Bachelor's or College Degree,Not have,Not Applicable,Career shifter - currently working in a non-da...,...,None Of The Above,No side gig p12m,4.0,Not applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-eed25f,Balance Luzon,20 to 24,35k and less
1,2,Philippines,Binan,Not Applicable,Male,36,Master's degree,Not have,Not Applicable,Professional - employed full-time in a data-re...,...,Data Analysis & Insights,No side gig p12m,4.0,4 to 5,"Yes, I use AI tools WEEKLY for work.","Yes, I use AI tools WEEKLY for study",2025-c107bd,Balance Luzon,35 to 39,35k and less
2,3,Philippines,Mandaluyong,Not Applicable,Female,32,Bachelor's or College Degree,Have comp or data degree,BS Electronics and Communications Engineering,Professional - employed full-time in a data-re...,...,Data Analysis & Insights,No side gig p12m,6.0,30+,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools WEEKLY for study",2025-b053b8,Metro Manila,30 to 34,100k and above
3,4,Philippines,San Quintin,Not Applicable,Male,34,Bachelor's or College Degree,Not have,Not Applicable,Career shifter - currently working in a non-da...,...,None Of The Above,No side gig p12m,10.0,30+,"No, but I plan to soon.","Yes, I use AI tools DAILY for study.",2025-a03238,Balance Luzon,30 to 34,35k+ to 75k
4,5,Philippines,Taguig,Not Applicable,Male,29,Bachelor's or College Degree,Have comp or data degree,BS Applied Mathematics,Professional - employed full-time in a data-re...,...,Data Engineering,No side gig p12m,6.0,6 to 10,"Yes, I use AI tools DAILY for work.","No, but I plan to soon.",2025-ae4e76,Metro Manila,25 to 29,100k and above
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1856,1857,Philippines,Quezon City,Prefer Not To Say,Male,27,Bachelor's or College Degree,Not have,BS Information Technology,Student - currently studying and not yet worki...,...,Not Applicable,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools WEEKLY for work.","Yes, I use AI tools WEEKLY for study",2025-05787a,Metro Manila,25 to 29,Not Applicable
1857,1858,Philippines,Puerto Princesa,Prefer Not To Say,Female,28,Bachelor's or College Degree,Have comp or data degree,BS Computer Engineering,Looking for data-related work - actively seeki...,...,Not Applicable,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools MONTHLY or LESS frequently...","Yes, I use AI tools MONTHLY or LESS frequently...",2025-e0860f,Balance Luzon,25 to 29,Not Applicable
1858,1859,Philippines,Bohol,Not Applicable,Female,35,Bachelor's or College Degree,Have comp or data degree,BS Information Technology,Career shifter - currently working in a non-da...,...,Data Processing & Entry,No side gig p12m,4.0,4 to 5,"Yes, I use AI tools MONTHLY or LESS frequently...","No, but I plan to soon.",2025-0b73c1,Visayas,35 to 39,35k and less
1859,1860,Mexico,Sinaloa,Culiacan,Male,23,Bachelor's or College Degree,Not have,Not Applicable,Student - currently studying and not yet worki...,...,Not Applicable,Not Applicable,999.0,Not Applicable,"Yes, I use AI tools DAILY for work.","Yes, I use AI tools DAILY for study.",2025-571482,Not Applicable,20 to 24,Not Applicable


# 6j. Export single response with groups summary

In [3]:
# 6j. Export compilation as csv file
# Important for summary later

csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)
df_single_no_grps = pd.read_csv(csv_outputs_dir/"df_single_with_grps.csv")
single_cols_with_grps = [
    "resp_id", "country", "city_ph", "city_non_ph", "gender", "age", "educstat", "have_comp_degree", "comp_related_degree", "careerstg", "employertype", "industry", "salary", "how_long_in_salary", "typework", "sitework", "shift_schedule", "datarole", "side_gig", "satisfaction", "sizeteam", "ai_work", "ai_study", "resp_id_rand", "region_ph", "age_grp", "salary_broader"
]

parsed_rows = []  # <-- define this once before the loop, saved using pd.DataFrame

# Eda function also defined in 5g.
def explore_categorical_cols_to_csv(col, df):
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')
    print(f'Exploration of {col} column')
    print('~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~')

    # Header row for CSV
    parsed_rows.append({
        "column": col,
        "label": f"Description of {col} ~~~~~~~~~~",
        "value": ""
    })

    unique_count = df[col].nunique()

    if unique_count > 100:
        msg = f"Skipping {col} (unique values = {unique_count}, exceeds threshold of 100)."
        print(msg)
        parsed_rows.append({
            "column": col,
            "label": "skip_reason",
            "value": msg
        })
        print()
        return

    print(f'Description of {col} ~~~~~~~~~~')
    desc = df[col].describe()
    print(desc)
    print()

    # Parsed describe() rows
    for label, value in desc.items():
        parsed_rows.append({
            "column": col,
            "label": label,
            "value": value
        })

    print(f'Value counts of {col} ~~~~~~~~~~')
    vc = df[col].value_counts(dropna=False)
    print(vc)
    print()

    # Parsed value_counts() rows
    for label, value in vc.items():
        parsed_rows.append({
            "column": col,
            "label": label,
            "value": value
        })

# Apply eda function 
for col in single_cols_with_grps:
    explore_categorical_cols_to_csv(col, df_single_no_grps)

summary_df_single_with_grps = pd.DataFrame(parsed_rows)
summary_df_single_with_grps.to_csv(os.path.join(csv_outputs_dir, "summary_single_with_grps.csv"), index=False)


print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Exploration of resp_id column
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Skipping resp_id (unique values = 1861, exceeds threshold of 100).

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Exploration of country column
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Description of country ~~~~~~~~~~
count            1861
unique             31
top       Philippines
freq             1798
Name: country, dtype: object

Value counts of country ~~~~~~~~~~
country
Philippines       1798
Nigeria             11
Egypt                7
UAE                  5
Canada               3
United Kingdom       3
KSA                  3
United States        2
Nepal                2
Uganda               2
Ghana                2
Kenya                2
Peru                 2
South Africa         2
Pakistan             1
Jordan               1
Cambodia             1
Thailand             1
Norway               1
Cameroon             1
Malaysia             

# 7. Handle multi-response columns
- Define multi_response_cols list
- Create explode_and_lookup function

In [17]:
# 7. Define and filter multi-response columns

# Import df_raw if starting this section again
df_raw = pd.read_parquet(parquet_outputs_dir/"df_raw.parquet")
df_raw


,resp_id,country,city_ph,city_non_ph,gender,age,educstat,have_comp_degree,comp_related_degree,digitools,...,ai_study,ai_tools,hardware,dep_init,volunteer,spneeds,otherfb,attended_online,attended_inperson,resp_id_rand
0,1,Philippines,Pampanga,Not Applicable,Male,23,Bachelor's or College Degree,No computer-related or data-related degree,Not Applicable,Coursera;Datacamp;Youtube;Eskwelabs,...,"Yes, I use AI tools DAILY for study.",Chatgpt,desktop;laptop,DEP DataCamp scholarship;DEP DataMasters;DEP T...,Answer surveys;Give advice to people in the gr...,Be part of a community;Career advancement;Cont...,None of the above,Eskwelabs bootcamp and datamasters,Not Applicable,2025-eed25f
1,2,Philippines,Binan,Not Applicable,Male,36,Master's degree,No computer-related or data-related degree,Not Applicable,Datacamp;MicrosoftLearn;W3Schools;Youtube,...,"Yes, I use AI tools WEEKLY for study",Chatgpt;Copilot,laptop,DEP DataCamp scholarship;DEP The Puso Project;...,Answer surveys;Give advice to people in the gr...,Be part of a community;Career advancement;Cont...,Analytics Association of the Philippines;Commu...,Career Shifting and Data Analysis discord meeting,Not Applicable,2025-c107bd
2,3,Philippines,Mandaluyong,Not Applicable,Female,32,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Electronics and Communications Engineering,Cisco;Coursera;Datacamp;LinkedIn Learning;O'Re...,...,"Yes, I use AI tools WEEKLY for study",Chatgpt;Copilot;Gemini;Grok;Perplexity,laptop;VM,DEP DataCamp scholarship;DEP DataHub;DEP Study...,Answer surveys;Sponsor/Invest,Continued learning,Data Engineering Pilipinas,Not Applicable,Not Applicable,2025-b053b8
3,4,Philippines,San Quintin,Not Applicable,Male,34,Bachelor's or College Degree,No computer-related or data-related degree,Not Applicable,Cisco;Coursera;Datacamp;Udemy;W3Schools;Youtube,...,"Yes, I use AI tools DAILY for study.",Chatgpt;Cici;Copilot,desktop;laptop,DEP DataCamp scholarship;DEP youtube channel,Answer surveys;Share knowledge;Teach soft skills,Be part of a community;Career advancement;Cont...,Datasense Philippines,Not Applicable,None,2025-a03238
4,5,Philippines,Taguig,Not Applicable,Male,29,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Applied Mathematics or Related,AWS Educate;Coursera;Udemy,...,"No, but I plan to soon.",Chatgpt;Claude;Copilot;Cursor;Gemini;Github co...,laptop;VM;workstation,DEP DataCamp scholarship;DEP DataMasters;DEP d...,Answer surveys;Give advice to people in the gr...,Be part of a community;Continued learning;Network,Analytics Association of the Philippines;AWS U...,discord event,Not Applicable,2025-ae4e76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1856,1857,Philippines,Quezon City,Prefer Not To Say,Male,27,Bachelor's or College Degree,No computer-related or data-related degree,BS Information Technology,Datacamp,...,"Yes, I use AI tools WEEKLY for study",Chatgpt;Claude;Gemini;Humanize.ai;Perplexity;v...,desktop,None of the above,None of the above,Career advancement;Continued learning,IT Philippines,none,none,2025-05787a
1857,1858,Philippines,Puerto Princesa,Prefer Not To Say,Female,28,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Computer Engineering,Coursera;Datacamp;MicrosoftLearn,...,"Yes, I use AI tools MONTHLY or LESS frequently...",Copilot,laptop,None of the above,Answer surveys;Co-training/ Training;Share kno...,Be part of a community;Career advancement;Upsk...,Data Analytics Philippines,FREE WEBINAR MOST OF THE TIME,"NONE, CONDUCTED VIA ONLINE ONLY",2025-e0860f
1858,1859,Philippines,Bohol,Not Applicable,Female,35,Bachelor's or College Degree,"Yes, I have a computer-related or data-related...",BS Information Technology,Youtube,...,"No, but I plan to soon.",Copilot,laptop,DEP DataCamp scholarship,Answer surveys;Share insights to members;Share...,Continued learning;Decision to shift;Upskill /...,Analytics Association of th

In [18]:

# Define multi-response columns
multi_response_cols = ["digitools", "successmethod", "restofrole", "ingestion", "transform", "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools", "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds", "otherfb", "attended_inperson", "attended_online"]

# filter using multi-response columns
df_raw_multi = df_raw[['resp_id'] + multi_response_cols].copy()
df_raw_multi.to_csv(csv_outputs_dir/"df_raw_multi.csv", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")  )
df_raw_multi.columns

Date and Time:  2026-03-19 17:58:12


Index(['resp_id', 'digitools', 'successmethod', 'restofrole', 'ingestion',
       'transform', 'storage', 'warehs', 'orchest', 'busint', 'cloudplat',
       'generaltools', 'whatused', 'ai_tools', 'hardware', 'dep_init',
       'volunteer', 'spneeds', 'otherfb', 'attended_inperson',
       'attended_online'],
      dtype='object')

In [19]:
# 7a1. Remove comma in MS Fabric under ingestion
# print (df_raw_multi['ingestion'].unique())

# Replace the MS Fabric item
df_raw_multi['ingestion'] = df_raw_multi['ingestion'].str.replace("MS Fabric (Eventstreams, Dataflow, COPY, etc.)", "MS Fabric (Evenstreams Dataflow COPY)")

# Inspect again
# Filter the column first, then get unique values
ms_fabric_ingestion = df_raw_multi[df_raw_multi['ingestion'].str.contains("MS Fabric")]['ingestion']
print(ms_fabric_ingestion.unique())
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

['MS Excel;MS Fabric (Evenstreams Dataflow COPY);MS Power Platform;Power BI'
 'Azure Data Factory;MS Fabric (Evenstreams Dataflow COPY);MSSQL replication;Power BI;SSIS'
 'Alteryx;Amazon Kinesis;Apache Kafka;AWS Glue;AWS Lambda;Azure Data Factory;Azure Databricks;Azure Synapse Analytics;Databricks;Dataverse;Fivetran;Google Sheets;Google Cloud Composer;Google Cloud Dataflow;Hadoop;Knime;MS Excel;MS Fabric (Evenstreams Dataflow COPY);MS Forms;MS Power Platform;MSSQL replication;Oracle;Power BI;Python scripts;Salesforce;Snowflake;Spark;SQL;SSIS;Tableau Prep;Talend'
 'Azure Data Factory;Azure Databricks;Azure Synapse Analytics;Databricks;MS Fabric (Evenstreams Dataflow COPY);Power BI;Python scripts;Snowflake;Spark;SQL'
 'AWS Glue;Azure Databricks;MS Fabric (Evenstreams Dataflow COPY);MS Power Platform;Power BI;SQL;Tableau Prep'
 'Google Sheets;MS Excel;MS Fabric (Evenstreams Dataflow COPY);MS Forms;MS Power Platform;Tableau Prep'
 'AWS Glue;Azure Databricks;Databricks;Google Sheets;MS Excel

In [20]:
# 7a2. Remove comma in MS Fabric under orchestration
# print (df_raw_multi['orchest'].unique())

# Replace the MS Fabric item under orchestration
df_raw_multi['orchest'] = df_raw_multi['orchest'].str.replace("MS Fabric (Pipelines, etc.)", "MS Fabric (Pipelines)")

# Inspect again
ms_fabric_orchestration = df_raw_multi[df_raw_multi['orchest'].str.contains("MS Fabric")]['orchest']
print(ms_fabric_orchestration.unique())
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


['MS Fabric (Pipelines);SQL Server Agent'
 'Airflow;Astronomer;AWS Step Functions;Azure Data Factory;Cron;Databricks Workflows;GCP Cloud Workflows;Kubeflow;MS Fabric (Pipelines);MS Power Automate;SQL Server Agent;Windows Task Scheduler'
 'Airflow;MS Fabric (Pipelines);MS Power Automate'
 'AWS Step Functions;MS Fabric (Pipelines)' 'MS Fabric (Pipelines)'
 'Azure Data Factory;Databricks Workflows;MS Fabric (Pipelines)'
 'Databricks Workflows;MS Fabric (Pipelines);MS Power Automate'
 'MS Fabric (Pipelines);MS Power Automate'
 'Databricks Workflows;Kubeflow;MS Fabric (Pipelines)'
 'Azure Data Factory;MS Fabric (Pipelines);Orkes'
 'Azure Data Factory;MS Fabric (Pipelines)'
 'MS Fabric (Pipelines);MS Power Automate;SQL Server Agent'
 'Azure Data Factory;MS Fabric (Pipelines);MS Power Automate'
 'MS Fabric (Pipelines);MS Power Automate;Windows Task Scheduler;None of the above'
 'Azure Data Factory;MS Fabric (Pipelines);MS Power Automate;Sematic;SQL Server Agent;Windows Task Scheduler'
 'Azure

In [21]:
# 7a3. Remove comma in MS Fabric under transformation
# print (df_raw_multi['transform'].unique())

# Replace the MS Fabric item under transformation
df_raw_multi['transform'] = df_raw_multi['transform'].str.replace("MS Fabric (dbt-fabric, Dataflow, etc.)", "MS Fabric (dbt-fabric Dataflow)")

# Inspect again
ms_fabric_transformation = df_raw_multi[df_raw_multi['transform'].str.contains("MS Fabric")]['transform']
print(ms_fabric_transformation.unique())
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


['MS Fabric (dbt-fabric Dataflow);MS Power BI Dataflows;SQL-Based Platforms'
 'Alteryx;AWS Athena;AWS Glue;Azure Data Factory;Dataverse;Google BigQuery;Google Sheets;Informatica;Knime;MS Excel;MS Fabric (dbt-fabric Dataflow);MS Power Apps;Pandas;Polars;Postgresql;MS Power Automate;MS Power Query;MS Power BI Dataflows;Pyspark;Python;R;Spark;SQL-Based Platforms;SQL/Full SQL(BigQuery);SQL/PL/SQL(Oracle);SQL/Postgresql;SQL/Snowflake;SQL/SparkSQL(Databricks);SQL/T-SQL;SSIS;Tableau Prep;XSLT'
 'MS Excel;MS Fabric (dbt-fabric Dataflow);MS Power Apps;MS Power Automate;MS Power Query;MS Power BI Dataflows'
 'Azure Data Factory;MS Fabric (dbt-fabric Dataflow);Pyspark;Spark;SQL/Snowflake;SQL/SparkSQL(Databricks)'
 'Azure Data Factory;MS Fabric (dbt-fabric Dataflow);Python;SQL-Based Platforms'
 'Google Sheets;Informatica;MS Excel;MS Fabric (dbt-fabric Dataflow);MS Power Apps;R'
 'Azure Data Factory;MS Excel;MS Fabric (dbt-fabric Dataflow);SQL/Postgresql;SQL/T-SQL'
 'Azure Data Factory;Dataverse;MS

In [22]:
# 7b. Special cleaning for attended inperson and online - clean and standardize None responses
# Using fuzzy match
cols_attended = ['attended_inperson', 'attended_online']

for col in cols_attended:

    # Fuzzy match patterns
    mask_fuzzy = df_raw_multi[col].str.lower().str.contains(
        r'non|none|nope|n/a',
        na=False
    )

    # Exact none-like values
    mask_exact = df_raw_multi[col].str.strip().str.lower().isin([
        'nan',
        '/a',
        'n/a',
        'nope',
        'no e',
        'npne',
        'nine',
        'nin',
        'n n'
    ])

    # True NaN values
    mask_nan = df_raw_multi[col].isna()

    # Combine all masks
    mask_combined = mask_fuzzy | mask_exact | mask_nan

    # Replace with standard value
    df_raw_multi.loc[mask_combined, col] = 'None'

    # Final normalization
    df_raw_multi[col] = df_raw_multi[col].str.strip().str.upper()

    # Print unique values
    unique_list_attended= sorted(set(df_raw_multi[col].to_list()))
    print(f"Unique values in {col}")
    print("~"*40)
    print(unique_list_attended)
    print("~"*40, "\n\n")

# ---------------------------------------------
# df_raw_multi saved as df_multi after cleaning
# ---------------------------------------------
df_multi = df_raw_multi.copy()

# Export df_raw_multi to csv
df_multi.to_csv(csv_outputs_dir/"df_multi.csv", index = False)  
df_multi.to_parquet(parquet_outputs_dir/"df_multi.parquet", index = False, engine="pyarrow")
print ("df_raw_multi is saved as df_multi after initial cleaning.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Unique values in attended_inperson
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
['"NEXT-GEN JAVA: AI, CLOUD, & PERFORMANCE" SEMINAR', '2025 PHILIPPINE JUNIOR DATA SCIENCE CHALLENGE', '2ND MEET UP IN CBTL', 'AI HACKATHON, NEXTGENPH, STUDENT CONGRESS', 'AIBP CONVENTION, DATA STREAMS WORKSHOPS', 'ATTENDED A SEMINAR ORGANIZED BY AI PILIPINAS', 'AWS CLOUD DAY', 'AWS COMMUNITY DAY', 'AWS INNOVATION CUP', 'AWS STUDENT COMMUNITY NIGHT', 'BIBE', 'BIG DATA EUROPE', 'BPI DATA WAVE, DICT PSC', 'BPI WAVE HACKATHON', 'CTRL ALT RUN', 'DATA ENGINEERING BULACAN MEETUP', 'DATA FUTURE PHILIPPINES 2025', 'DAXDAKAN', 'DAXDAKAN LIVE', 'DEEPLEARNING.AI HACKATHON', 'DEP DECEMBER CONFERENCE', 'DEP SOUTH MEETUP', 'DEVCON', 'DEVCON MORE ABOUT ON AI', 'DICT STARTUP CONTEST', 'DSA ONLINE TRAINING', 'EHEADS', 'EHEADS, PYCON APAC 2025, AI PIE MANILA', 'EHEADS20', 'FEU TECH CS EXPO', 'FREE WEBINARS, SHORT COURSES ONLINE', 'FTW FOUNDATION DATA ANALYTICS PROGRAM', 'GDG GOOGLE DEV SEMINARS AND WORKSHOPS', 'GOOGLE DATA ANALY

In [ ]:
df_multi

In [ ]:
# 7c. Transform and normalize string values, explode and normalize functions for multi-response columns
# Inspect the df_multi columns.  If delimiter is semi-colon, retain in function. Otherwise, replace with comma.

# Transform string values (nested function inside explode and normalize)
def normalize_text(val):
    if pd.isna(val):
        return ""
    val = unicodedata.normalize("NFKC", str(val))
    val = val.strip()
    val = " ".join(val.split())
    return val


def explode_and_normalize(df, column):
    # Include resp_id and multi-response columns
    exploded = (
        df[["resp_id", column]]
        .assign(**{column: df[column].str.split(";")})  # Split by semi-colon
        .explode(column)
    )
    exploded[column] = exploded[column].apply(normalize_text)

    # Remove duplicates (resp_id, value) pairs
    exploded = exploded.drop_duplicates(subset=["resp_id", column], keep="first")

    exploded.to_parquet(parquet_outputs_dir / f"{column}_exploded.parquet", index = False, engine="pyarrow")
    exploded.to_csv(csv_outputs_dir/f"{column}_exploded.csv", index = False)
    return exploded

print("Functions normalize_text and explode and normalize created")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Functions normalize_text and explode and normalize created
Date and Time:  2026-03-19 18:01:19


In [ ]:
# 7d. Apply explode_and_normalize function to (cleaned) df_multi subset → csv, parquet

# Initialize the df_multi csv file
df_multi.to_csv(csv_outputs_dir / "df_multi.csv", index=False)

# Explode each multi-response column
for col in multi_response_cols:
    # Explode and lookup for this column
    df_exploded = explode_and_normalize(df_multi, col)

    # Export exploded dataframe as {col}_exploded.csv and parquet
    df_exploded.to_csv(csv_outputs_dir / f"{col}_exploded.csv", index=False)
    df_exploded.to_parquet(parquet_outputs_dir / f"{col}_exploded.parquet", index=False, engine="pyarrow")

print("Multi-response columns exploded → csv, parquet.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Multi-response columns exploded → csv, parquet.
Date and Time:  2026-03-19 18:01:34


In [27]:
# 7e. Special cleaning for datarole and restofrole 
# Check datarole duplicates by merging datarole and restofrole
# If datarole and restofrole are the same, then they are duplicates
# Create helper column
df_restofrole_exploded = pd.read_csv(csv_outputs_dir/"restofrole_exploded.csv")
df_single_with_grps = pd.read_csv(csv_outputs_dir/"df_single_with_grps.csv")
df_datarole = df_single_with_grps[['resp_id', 'datarole']]
df_role_merged = df_datarole.merge(df_restofrole_exploded, on = 'resp_id', how = 'inner')
df_role_merged['duplicated'] = (
    df_role_merged['datarole'].str.strip().str.lower()
    == df_role_merged['restofrole'].str.strip().str.lower()
)


# Drop duplicated rows in df_role_merged
df_role_merged = df_role_merged[df_role_merged['duplicated'] == False]

# Drop the helper column
df_role_merged = df_role_merged.drop(columns=['duplicated'])
df_role_merged.to_csv(csv_outputs_dir/"df_role_merged.csv", index = False)


# Create an updated restofrole exploded csv with a longer name and shorter name - for general cleaning later
df_restofrole_before_general_cleaning = df_role_merged[['resp_id', 'restofrole']].copy()
df_restofrole_before_general_cleaning.to_csv(csv_outputs_dir/"restofrole_exploded_before_general_cleaning.csv", index = False)
df_restofrole_before_general_cleaning.to_csv(csv_outputs_dir/"restofrole_exploded.csv", index = False)

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Special cleaning of restofrole done.")

Date and Time:  2026-03-19 18:04:12
Special cleaning of restofrole done.


In [ ]:
# 7f. Second round of clean up multi-response columns after being exploded, especially the None variants

multi_response_cols = ["digitools", "successmethod", "restofrole", "ingestion", "transform", "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools", "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds", "otherfb", "attended_inperson", "attended_online"]

none_variants = [
    "None", "None ", "Non", "non", "non ", "npne", "n ne", "npn", "nine", "nin", "nond", "nonf"]

for col in multi_response_cols:
    path = csv_outputs_dir / f"{col}_exploded.csv"
    df_exp_cln = pd.read_csv(path)

    # Convert to string, strip spaces
    df_exp_cln[col] = df_exp_cln[col].astype("string").str.strip()

    # First, standardize blank-like values replace with pd.NA
    df_exp_cln[col] = df_exp_cln[col].replace(
        ["", "nan", "NaN"], pd.NA
    )

    # Then, normalize None-like strings to "None"
    df_exp_cln[col] = df_exp_cln[col].replace(none_variants, "None")

    # Drop rows where the exploded value is NA
    df_exp_cln = df_exp_cln.dropna(subset=[col])

    # Drop blanks
    df_exp_cln = df_exp_cln[df_exp_cln[col] != ''    ]

    # Save cleaned version
    df_exp_cln.to_csv(csv_outputs_dir / f"{col}_exploded_cleaned.csv", index=False)
    df_exp_cln.to_parquet(parquet_outputs_dir / f"{col}_exploded_cleaned.parquet", index=False, engine="pyarrow")

    # Save pre-cleaning version for attended tables for more cleaning later
    if col in ['attended_inperson', 'attended_online']:
        df_exp_cln.to_csv(csv_outputs_dir / f"{col}_exploded_cleaned_pre.csv", index=False)

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Second round of cleaning of multi-response columns done.")
print("Exploded and cleaned versions saved in csv and parquet.")

Date and Time:  2026-03-19 18:04:32
Second round of cleaning of multi-response columns done.


In [41]:
# 7g. Special cleaning for attended_online_exploded_cleaned.csv

# Define substring-based soft-matching themes
# PRIORITY ORDER MATTERS
# -------------------------------------------------------------------
themes_online = {
    "datamasters_specified": ["datam", "data masters", "data asters", "resume", "mock"],
    "webinars_seminars": ["webin", "webnar", "webminar", "webninar", "seminar", "pokemon"],
    "meetups": ["meetup"],
    "self_paced_learning": ["coursera", "udemy", "online course", "online courses"],
    "datasense_analytics": ["datasense", "datasencw", "data sense", "dsa", "avila"],        
    "dep_community": ["dep", "kamustahan", "survey results"],
    "datacamp_learning": ["camp"],
    "power_bi_pilipinas": ["power bi", "powerbi", "daxdakan"],
    "ai_ml": ["ai", "deep", "ml"],
    "school_university_events": ["school", "univ", "upd", "uplb"],
    "certifications_bootcamps": ["cert", "boot"],    
    "youtube_learning": ["yout"],
    "discord_events_datamasters_webinars": ["disc", "kumustahan"],
    "none": ["none", "not applicable", "y"]
}

# df_online = pd.read_csv(csv_outputs_dir/"attended_online_exploded_cleaned.csv")
df_online = pd.read_csv(csv_outputs_dir/"attended_online_exploded_cleaned_pre.csv")

# Normalize text for matching
df_online["attended_online_norm"] = (
    df_online["attended_online"]
    .astype(str)
    .str.lower()
    .str.strip()
)


# Create function assign_theme()
def assign_theme(text):
    for theme, substrings in themes_online.items():
        for sub in substrings:
            if sub in text:
                return theme
    return "other_single_mentions"

df_online["theme"] = df_online["attended_online_norm"].apply(assign_theme)

# Replace the original messy text with the theme label
df_online["attended_online"] = df_online["theme"].str.upper()

# Drop helper columns after copying above
df_online = df_online.drop(columns=["attended_online_norm", "theme"])

# Save the final cleaned version back to _exploded_cleaned.csv
# -------------------------------------------------------------------
df_online.to_csv(csv_outputs_dir/"attended_online_exploded_cleaned.csv", index=False)


print("Check if other responses below to other themes.  If none, this is final.")
print(" Original: csv_outputs_dir/attended_online_exploded_cleaned_pre.csv")
print(" Saved: csv_outputs_dir/ attended_online_exploded_cleaned.csv (theme-labeled)")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Check if other responses below to other themes.  If none, this is final.
 Original: csv_outputs_dir/attended_online_exploded_cleaned_pre.csv
 Saved: csv_outputs_dir/ attended_online_exploded_cleaned.csv (theme-labeled)
Date and Time:  2026-03-21 12:27:31


In [42]:
# 7g1. Check if other responses belong to current themes
valcounts = df_online["attended_online"].value_counts(dropna=False).reset_index()
valcounts.columns = ["value", "count"]
print(valcounts.to_string(index=False))


                              value  count
                               NONE   1543
                  WEBINARS_SEMINARS    117
DISCORD_EVENTS_DATAMASTERS_WEBINARS     45
              OTHER_SINGLE_MENTIONS     35
                  DATACAMP_LEARNING     31
              DATAMASTERS_SPECIFIED     26
                      DEP_COMMUNITY     17
                 POWER_BI_PILIPINAS     13
                              AI_ML      9
                SELF_PACED_LEARNING      8
                DATASENSE_ANALYTICS      7
                            MEETUPS      5
                   YOUTUBE_LEARNING      3
           CERTIFICATIONS_BOOTCAMPS      2


In [50]:
# 7h. Special cleaning for attended_inperson_exploded_cleaned.csv

# Define substring classifications 
themes_inperson = {
    "pycon_python_events": ["pycon", "pyladies"],
    "soscon_events": ["soscon"],
    "dep_community": ["dep", "daxdakan", "data talks"],
    "hackathons": ["hackathon", "hackaton"],
    "google_dev_events": ["devfest", "google io", "gdg"],
    "ai_ml_events": ["ai pilipinas", "aibp", "ai pie", "ai "],
    "power_bi_pilipinas": ["power bi", "powerbi"],
    "aws_events": ["aws"],
    "data_science_engineering_events": ["data science", "data engineering", "pjdsc", "junior data science"],
    "techbayanihan_events": ["techbayanihan"],
    "government_tech_events": ["dict", "ict congress", "planning institute"],
    "university_events": ["up ", "university", "nec", "ftw", "capstone", "youth summit"],
    "meetups_general": ["meetup"],
    "r_user_group": ["r user group"],
    "online_learning_misplaced": ["online", "webinar", "datacamp", "dsa"],
    "eheads_events": ["eheads"],
    "none": ["none", "not applicable", "forgot", "same as above", "noje", "y"]
}


import pandas as pd

# Load
df_inperson = pd.read_csv(csv_outputs_dir/"attended_inperson_exploded_cleaned_pre.csv")

# Normalize
df_inperson["attended_inperson_norm"] = (
    df_inperson["attended_inperson"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Use the dictionary defined above
themes = themes_inperson

# Classification function
def assign_theme_inperson(text):
    for theme, substrings in themes.items():
        for sub in substrings:
            if sub in text:
                return theme
    return "other_single_mentions"  # fallback for uncategorized strings

# Apply
df_inperson["theme"] = df_inperson["attended_inperson_norm"].apply(assign_theme_inperson)

# Replace original text with theme
df_inperson["attended_inperson"] = df_inperson["theme"].str.upper()

# Drop helpers
df_inperson = df_inperson.drop(columns=["attended_inperson_norm", "theme"])

# Save final
df_inperson.to_csv(csv_outputs_dir/"attended_inperson_exploded_cleaned.csv", index=False)

print("Check if other responses below to other themes.  If none, this is final.")
print(" Original: csv_outputs_dir/attended_inperson_exploded_cleaned_pre.csv")
print(" Saved: csv_outputs_dir/ attended_inperson_exploded_cleaned.csv (theme-labeled)")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Check if other responses below to other themes.  If none, this is final.
 Original: csv_outputs_dir/attended_inperson_exploded_cleaned_pre.csv
 Saved: csv_outputs_dir/ attended_inperson_exploded_cleaned.csv (theme-labeled)
Date and Time:  2026-03-21 12:32:10


In [51]:
# 7h1. Check if other responses belong to current themes
valcounts_inperson= df_inperson["attended_inperson"].value_counts(dropna=False).reset_index()
valcounts_inperson.columns = ["value", "count"]
print(valcounts_inperson.to_string(index=False))

                          value  count
                           NONE   1769
          OTHER_SINGLE_MENTIONS     18
                     HACKATHONS     11
              UNIVERSITY_EVENTS      8
                  DEP_COMMUNITY      6
DATA_SCIENCE_ENGINEERING_EVENTS      6
            PYCON_PYTHON_EVENTS      6
                     AWS_EVENTS      5
      ONLINE_LEARNING_MISPLACED      5
         GOVERNMENT_TECH_EVENTS      4
                  EHEADS_EVENTS      4
              GOOGLE_DEV_EVENTS      4
                  SOSCON_EVENTS      3
             POWER_BI_PILIPINAS      3
           TECHBAYANIHAN_EVENTS      3
                   AI_ML_EVENTS      2
                MEETUPS_GENERAL      2


In [52]:
# 7i. Summary of cleaned multi-response columns
# Recall the multi-response columns definition
multi_response_cols = [
    "digitools", "successmethod", "restofrole", "ingestion", "transform",
    "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools",
    "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds",
    "otherfb", "attended_inperson", "attended_online",
]

# Initialize the summary_frames list for pd.concat later
summary_frames = []

# Build summary of cleaned multi-response columns
def build_summary_multi(col):
    for col in multi_response_cols:
        # Build summary for this column
        path = csv_outputs_dir / f"{col}_exploded_cleaned.csv"
        df_exploded = pd.read_csv(path)

        # Remove duplicates on (resp_id, col) pair, retain first
        df_exploded = df_exploded.drop_duplicates(subset=["resp_id", col], keep="first")

        value_counts = df_exploded[col].value_counts(dropna=False).reset_index()
        value_counts.columns = ["value", "count"]
        value_counts.insert(0, "column", col)
        summary_frames.append(value_counts)

        # Divider row
        divider = pd.DataFrame([{"column": "~" * 50, "value": "", "count": ""}])
        summary_frames.append(divider)

# Call build summary        
build_summary_multi(multi_response_cols)

# Combine all summaries and export
summary_combined = pd.concat(summary_frames, ignore_index=True)
summary_combined.to_csv(csv_outputs_dir / "summary_multi_response.csv", index=False)

print("Summary of exploded cleaned tables created.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


Summary of exploded cleaned tables created.
Date and Time:  2026-03-21 12:32:29


# 7i. Check duplicates (using raw strings not cleaned for more accurate assessment)
- merge df_possible_duplicates with multi-response column strings
- Pre-sort strings then concatenate, not the cleaned versions, but using raw versions

In [ ]:
#7i. Check similarity of df_possible_duplicates including multi-response columns
# Load dfs -> possible duplicates (single), merge with multi-response cols 
# -> split items list from string → "" blanks →  strip spaces  → sort → ", ".join to string
# Exploded files are unchanged.

df_possible_duplicates = pd.read_csv(csv_outputs_dir/"df_possible_duplicates.csv")
df_multi = pd.read_csv(csv_outputs_dir/"df_multi.csv")
df_likely_dups = pd.read_csv(csv_outputs_dir/"table_of_likely_dups.csv")

multi_response_cols = [
    "digitools", "successmethod", "restofrole", "ingestion", "transform",
    "storage", "warehs", "orchest", "busint", "cloudplat", "generaltools",
    "whatused", "ai_tools", "hardware", "dep_init", "volunteer", "spneeds",
    "otherfb", "attended_inperson", "attended_online"
]

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Merge df_possible_duplicates with df_multi FIRST (inner join)
# This keeps only respondents present in both tables
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

df_merged_dups = df_possible_duplicates.merge(
    df_multi[["resp_id"] + multi_response_cols],
    on="resp_id",
    how="inner"
)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Standardization function for multi-response columns
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def standardize_multi_response(s):
    if pd.isna(s) or s.strip() == "":
        return ""
    items = [x.strip() for x in s.split(",")]
    items = sorted([x for x in items if x != ""])
    return ", ".join(items)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Apply standardization ONLY to the merged subset
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

for col in multi_response_cols:
    df_merged_dups[col] = df_merged_dups[col].apply(standardize_multi_response)

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Build explicit duplicate pairs from df_likely_dups
# Each row has resp_id and a list like [577]
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

resp_b = df_likely_dups["best_match_resp_ids"].str.strip("[] ").astype(int)

df_pairs = pd.DataFrame({
    "resp_id_1": df_likely_dups["resp_id"].combine(resp_b, min),
    "resp_id_2": df_likely_dups["resp_id"].combine(resp_b, max),
}).drop_duplicates()


# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Compare multi-response columns for each pair
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

results = []


# Build once before the loop — O(1) lookup by resp_id instead of O(n) scan
lookup = df_merged_dups.set_index("resp_id")

for _, row in df_pairs.iterrows():
    id1, id2 = row["resp_id_1"], row["resp_id_2"]
    row_a = lookup.loc[id1]   # instant index lookup
    row_b = lookup.loc[id2]
    comparison = {
        "resp_id_1": id1,
        "resp_id_2": id2,
    }

    for col in multi_response_cols:
        comparison[col] = (row_a[col] == row_b[col])

    results.append(comparison)

# Convert results list to dataframe
df_multi_compare = pd.DataFrame(results)
df_multi_compare['count_true'] =df_multi_compare[multi_response_cols].sum(axis=1)
df_multi_compare['count_false'] = df_multi_compare.shape[1] - df_multi_compare['count_true']

df_multi_compare.to_csv(csv_outputs_dir/"df_multi_compare.csv", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Duplicate check using multi-response columns done.")


Date and Time:  2026-03-19 18:05:05
Duplicate check using multi-response columns done.


In [ ]:
# Check
df_multi_compare

In [31]:
# Full duplicate test
print(df_multi_compare[['resp_id_1', 'resp_id_2', 'count_true', 'count_false']])

print("~"*75)
print("If count_false > 0, then resp1 and resp2 are not duplicates.")
print("~"*75, "\n")

# Check for possible duplicates
if (df_multi_compare['count_false'] == 0).any():
    print("Possible duplicates found.")
else:
    print("No possible duplicates.")

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*75, "\n")


# Second run - multi-response columns are different. Nothing to delete.

    resp_id_1  resp_id_2  count_true  count_false
0         471        502          11           12
1        1190       1338          10           13
2        1123       1628          13           10
3         195       1189          12           11
4         239       1418          11           12
5         961       1419          12           11
6         845       1211          10           13
7         839       1285          10           13
8        1291       1831          13           10
9         218       1315          12           11
10        202        781          10           13
11       1534       1624          13           10
12        592       1120          12           11
13       1198       1736          13           10
14       1727       1728          11           12
15       1099       1574          11           12
16        198       1337          13           10
17        101        357          12           11
18        343       1382          10           13


# 8.  Lineage - location dfs

- df_loc_new <- df_single_with_groups
- df_ph_grp, df_non_ph_grp <- df_loc_new 
- overseas <- df_ph_grp 
- df_ph_geo_clean <- df_ph_geo
- df_non_ph_geo_clean <- df_non_ph_geo
- df_all_geo_clean <- df_ph_geo_clean, df_non_ph_geo_clean

In [153]:
# 8. Load df_loc_new and save
# ~~~~~~~~~~~~~~~~~~~~~~~~~
df_single_with_grps = pd.read_csv(csv_outputs_dir/ "df_single_with_grps.csv")
df_loc_new = df_single_with_grps[["resp_id", "country", "city_ph", "city_non_ph"]].drop_duplicates()

location_dir = Path("location_dir")
location_dir.mkdir(exist_ok = True)
df_loc_new.to_csv(location_dir/ "df_loc_new.csv", index = False)

df_loc_new.info()
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1861 entries, 0 to 1860
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   resp_id      1861 non-null   int64 
 1   country      1861 non-null   object
 2   city_ph      1861 non-null   object
 3   city_non_ph  1861 non-null   object
dtypes: int64(1), object(3)
memory usage: 58.3+ KB
Date and Time:  2026-03-17 16:09:39


In [154]:
# 8a. Check non-ph cities
# If Prefer Not To Say or Not Applicable change to capital city later using lookup
not_in_ph = df_loc_new[df_loc_new['country'] != "Philippines"]
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
not_in_ph



Date and Time:  2026-03-17 16:10:06
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,resp_id,country,city_ph,city_non_ph
5,6,United Kingdom,Not Applicable,Newcastle
20,21,United Kingdom,Not Applicable,London
157,158,Norway,Not Applicable,Oslo
183,184,Cameroon,Not Applicable,Douala
232,233,Cambodia,Not Applicable,Phnom Penh
...,...,...,...,...
1837,1838,Taiwan,Not Applicable,Prefer Not To Say
1843,1844,Nigeria,Not Applicable,Nigeria
1852,1853,Nigeria,Not Applicable,Lagos
1853,1854,Nigeria,Not Applicable,Lagos


In [155]:
# 8c. Check ph cities
df_ph_grp = df_loc_new[df_loc_new['country'] == "Philippines"].groupby(['country','city_ph', 'city_non_ph'])['resp_id'].count().reset_index()
df_ph_grp.rename(columns = {"resp_id": "count_resp"}, inplace = True)
df_ph_grp.to_csv(location_dir/"df_ph_grp.csv", index=False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
df_ph_grp

Date and Time:  2026-03-17 16:10:15
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_ph,city_non_ph,count_resp
0,Philippines,Aklan,Not Applicable,1
1,Philippines,Albay,Not Applicable,11
2,Philippines,Aleosan,Not Applicable,1
3,Philippines,Angeles,Not Applicable,1
4,Philippines,Angono,Not Applicable,3
...,...,...,...,...
182,Philippines,Virac,Not Applicable,1
183,Philippines,Western Samar,Not Applicable,1
184,Philippines,Zambales,Not Applicable,5
185,Philippines,Zambales,Prefer Not To Say,1


In [156]:
# 8d. Cannot force city_non_ph to be "Not Applicable" if country = Philippines
# Lots of 'Prefer Not To Say', likely working overseas

overseas = df_ph_grp[df_ph_grp['city_non_ph'] == "Prefer Not To Say"]
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
overseas

Date and Time:  2026-03-17 16:10:24
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_ph,city_non_ph,count_resp
6,Philippines,Antipolo,Prefer Not To Say,1
18,Philippines,Batangas,Prefer Not To Say,1
31,Philippines,Butuan,Prefer Not To Say,1
35,Philippines,Cagayan,Prefer Not To Say,1
37,Philippines,Cagayan De Oro,Prefer Not To Say,2
43,Philippines,Caloocan,Prefer Not To Say,3
52,Philippines,Cavite,Prefer Not To Say,1
54,Philippines,Cebu,Prefer Not To Say,1
73,Philippines,Iligan City,Prefer Not To Say,1
100,Philippines,Mandaluyong,Prefer Not To Say,2


In [158]:
# 8e. Check city_non_ph if complete for geocoding
df_non_ph_grp = df_loc_new[df_loc_new['country'] != "Philippines"].groupby(['country','city_non_ph', 'city_ph'])['resp_id'].count().reset_index()
df_non_ph_grp = df_non_ph_grp.rename(columns={"resp_id":"count_resp"})
df_non_ph_grp.to_csv(location_dir/"df_non_ph_grp_before_lookup.csv", index = False)
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
df_non_ph_grp

Date and Time:  2026-03-17 16:11:59
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_non_ph,city_ph,count_resp
0,Algeria,Prefer Not To Say,Not Applicable,1
1,Bangladesh,Barishal,Not Applicable,1
2,Cambodia,Phnom Penh,Not Applicable,1
3,Cameroon,Douala,Not Applicable,1
4,Canada,Prefer Not To Say,Caloocan,1
5,Canada,Prefer Not To Say,Not Applicable,1
6,Canada,Regina,Not Applicable,1
7,Congo,Kinshasa,Not Applicable,1
8,Egypt,Alexandria,Not Applicable,2
9,Egypt,Cairo,Not Applicable,4


In [160]:
# 8f. Capital lookup for missing city_non_ph
import pandas as pd
df_non_ph_grp = pd.read_csv(location_dir/"df_non_ph_grp_before_lookup.csv")

# Extract all unique countries
unique_countries = sorted(df_non_ph_grp["country"].dropna().unique())

# Lookup dictionary only for missing cities
capital_lookup = {
    "Algeria": "Algiers",
    "Bangladesh": "Dhaka",
    "Cambodia": "Phnom Penh",
    "Cameroon": "Yaounde",
    "Canada": "Ottawa",
    "Congo": "Kinshasa",
    "Egypt": "Cairo",
    "Ghana": "Accra",
    "India": "New Delhi",
    "Jordan": "Amman",
    "Kenya": "Nairobi",
    "KSA":"Riyadh",
    "Malaysia": "Kuala Lumpur",
    "Mexico": "Mexico City",
    "Nepal": "Kathmandu",
    "Nigeria": "Abuja",
    "Norway": "Oslo",
    "Pakistan": "Islamabad",
    "Peru": "Lima",
    "Saudi Arabia": "Riyadh",
    "South Africa": "Pretoria",
    "South Korea": "Seoul",
    "Spain": "Madrid",
    "Taiwan": "Taipei",
    "Thailand": "Bangkok",
    "Uae": "Abu Dhabi",
    "UAE": "Abu Dhabi",
    "Uganda": "Kampala",
    "United Arab Emirates": "Abu Dhabi",
    "United Kingdom": "London",
    "United States": "Washington",
    "Vietnam": "Hanoi",
    "Yemen": "Sanaa",
}

# Find missing countries
missing = [c for c in unique_countries if c not in capital_lookup]

if len(missing) == 0:
    print("All countries are in the lookup dictionary.")
else:
    print ("Some countries are missing from the lookup")
    for c in missing:
        print("-", c)

# Stop here and manually update capital_lookup
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)


All countries are in the lookup dictionary.
Date and Time:  2026-03-17 16:15:19
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [161]:
# 8g. Replace missing or suppressed city_non_ph values with capital
# Geocode non ph cities -> wait 35 mins or less

df_non_ph_grp["city_non_ph"] = df_non_ph_grp.apply(
    lambda row: capital_lookup[row["country"]]
    if row["city_non_ph"] in ["Not Applicable", "Prefer Not To Say"]
    else row["city_non_ph"],
    axis=1
)

df_non_ph_grp.to_csv(location_dir/"df_non_ph_grp_after_lookup.csv", index = False)
df_non_ph_grp.to_csv(location_dir/"df_non_ph_grp.csv", index = False)
print("Missing values replaced with capitals")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("~"*50)
df_non_ph_grp

Missing values replaced with capitals
Date and Time:  2026-03-17 16:16:31
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_non_ph,city_ph,count_resp
0,Algeria,Algiers,Not Applicable,1
1,Bangladesh,Barishal,Not Applicable,1
2,Cambodia,Phnom Penh,Not Applicable,1
3,Cameroon,Douala,Not Applicable,1
4,Canada,Ottawa,Caloocan,1
5,Canada,Ottawa,Not Applicable,1
6,Canada,Regina,Not Applicable,1
7,Congo,Kinshasa,Not Applicable,1
8,Egypt,Alexandria,Not Applicable,2
9,Egypt,Cairo,Not Applicable,4


# Geospatial API call 1.non ph, 2. ph cities → combine
- Manually clean the final concatenated geo file, not upstream files



In [162]:
# 8h. Geocode non ph cities
# ~~~~~~~~~~~~~~~~~~~~~~~~~
# Import geopy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize geocoder
geolocator = Nominatim(user_agent="non_ph_geocoder")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# Extract unique cities for geocoding
unique_cities = df_non_ph_grp["city_non_ph"].dropna().unique()

# Prepare list for results
geo_results = []

for city in unique_cities:
    query = f"{city}"
    location = geocode(query)

    if location:
        geo_results.append(
            {
                "city_non_ph": city,
                "latitude": location.latitude,
                "longitude": location.longitude
            }
        )
    else:
        geo_results.append(
            {
                "city_non_ph": city,
                "latitude": None,
                "longitude": None
            }
        )

# Convert geocoding results to dataframe
df_geo = pd.DataFrame(geo_results)

# Merge coordinates back into main dataframe
df_non_ph_geo = df_non_ph_grp.merge(df_geo, on="city_non_ph", how="left")
df_non_ph_geo = df_non_ph_geo.rename(columns={"resp_id": "count_resp"})

# Export final geocoded file
df_non_ph_geo.to_csv(location_dir/"df_non_ph_geo.csv", index=False)

print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Need human check because of geopy mistakes.")
print("~"*50)
df_non_ph_geo


Date and Time:  2026-03-17 16:18:01
Need human check because of geopy mistakes.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_non_ph,city_ph,count_resp,latitude,longitude
0,Algeria,Algiers,Not Applicable,1,36.772933,3.058845
1,Bangladesh,Barishal,Not Applicable,1,22.493403,90.354801
2,Cambodia,Phnom Penh,Not Applicable,1,11.573039,104.857807
3,Cameroon,Douala,Not Applicable,1,4.042939,9.706202
4,Canada,Ottawa,Caloocan,1,45.420878,-75.690111
5,Canada,Ottawa,Not Applicable,1,45.420878,-75.690111
6,Canada,Regina,Not Applicable,1,50.447973,-104.615876
7,Congo,Kinshasa,Not Applicable,1,-4.319698,15.342420
8,Egypt,Alexandria,Not Applicable,2,44.834953,8.745030
9,Egypt,Cairo,Not Applicable,4,30.044388,31.235726


In [165]:
# 8i. Geocode ph cities
# The whole process takes 4-5 mins

# Import geopy
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

df_ph_grp = pd.read_csv(location_dir/"df_ph_grp.csv")

# Initialize geocoder
geolocator = Nominatim(user_agent="ph_city_geocoder")

# Rate limiter: 1 request per second
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.5)

# Extract unique PH cities
unique_cities = df_ph_grp["city_ph"].dropna().unique()

# Prepare list for results
geo_results = []

for city in unique_cities:
    query = f"{city}, Philippines"
    location = geocode(query)

    if location:
        geo_results.append(
            {
                "city_ph": city,
                "latitude": location.latitude,
                "longitude": location.longitude,
            }
        )
    else:
        geo_results.append(
            {
                "city_ph": city,
                "latitude": None,
                "longitude": None,
            }
        )

# Convert geocoding results to dataframe
df_geo = pd.DataFrame(geo_results)

# Merge coordinates back into grouped dataframe
df_ph_geo = df_ph_grp.merge(df_geo, on="city_ph", how="left")

# Export final geocoded file
df_ph_geo.to_csv(location_dir / "df_ph_geo.csv", index=False)


print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Need human check because of geopy mistakes.")
print("~"*50)
df_ph_geo


Date and Time:  2026-03-17 17:18:07
Need human check because of geopy mistakes.
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


,country,city_ph,city_non_ph,count_resp,latitude,longitude
0,Philippines,Aklan,Not Applicable,1,11.629789,122.248118
1,Philippines,Albay,Not Applicable,11,13.216667,123.550000
2,Philippines,Aleosan,Not Applicable,1,7.158581,124.577400
3,Philippines,Angeles,Not Applicable,1,15.138935,120.587532
4,Philippines,Angono,Not Applicable,3,14.525848,121.152968
...,...,...,...,...,...,...
182,Philippines,Virac,Not Applicable,1,13.580359,124.231203
183,Philippines,Western Samar,Not Applicable,1,11.833333,125.000000
184,Philippines,Zambales,Not Applicable,5,15.230000,120.120000
185,Philippines,Zambales,Prefer Not To Say,1,15.230000,120.120000


In [169]:
# 8i. Clean and combine ph and non ph dfs

# Create cleaned files for folium
df_ph_geo_clean = df_ph_geo.dropna(subset=["latitude", "longitude"])
df_ph_geo_clean.to_csv(location_dir/ "df_ph_geo_clean.csv", index = False)

df_non_ph_geo_clean = df_non_ph_geo.dropna(subset=["latitude", "longitude"])
df_non_ph_geo_clean.to_csv(location_dir/ "df_non_ph_geo_clean.csv", index = False)

df_ph_geo_clean = pd.read_csv(location_dir/"df_ph_geo_clean.csv")
df_non_ph_geo_clean = pd.read_csv(location_dir/"df_non_ph_geo_clean.csv")

# Enforce identical column order and names
required_cols = [
    "country",
    "city_ph",
    "city_non_ph",
    "count_resp",
    "latitude",
    "longitude"
]

df_ph_geo_clean= df_ph_geo_clean[required_cols]
df_non_ph_geo_clean = df_non_ph_geo_clean[required_cols]

# Concatenate
df_all_geo_clean = pd.concat([df_ph_geo_clean, df_non_ph_geo_clean], axis=0, ignore_index=True)

# Export final combined file
df_all_geo_clean.to_csv(location_dir/"df_all_geo_clean.csv", index=False)

print("Ph and Non-Ph geocoded files combined.")
print("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Manually clean the final concatenated geo file, not upstream files
# Zamboanga is located at latitude 6.91028 and longitude 122.07389.


Ph and Non-Ph geocoded files combined.
Date and Time:  2026-03-17 17:35:02


In [173]:
# 8j. Use folium to create an interactive map for df_all_geo_clean
# df_all_geo_clean → location_map_2026.html
import folium 
import pandas as pd

df_all_geo_clean = pd.read_csv("location_dir/df_all_geo_clean.csv")
# Create a map centered on the Philippines
m = folium.Map(location=[12.8797, 121.7740], zoom_start=6)

# Add markers for each city
for _, row in df_all_geo_clean.iterrows():

    # Choose the correct label
    label = row["city_ph"] if row["country"] == "Philippines" else row["city_non_ph"]

    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=label,
        tooltip=label,
        zoom_start  = 6,
        tiles = "CartoDB Positron"
    ).add_to(m)

# Display the map  (Optional - need to restart after Powershell command.))
# m   

# 8k. Using m.save to save the map
m.save("location_dir/location_map_2026.html")

In [171]:
# Visually inspect anomalies in location_map_2026.html
# Replace erroneous latitude and longitude for : 
#   Zamboanga, 6.91028, 122.07389
#   Quezon province,  13.9358, 121.6129
#   Egypt, Alexandria, 31.205753,29.924526
#   United States,Alexandria,1,38.820450,-77.050552
#   Peru,Not Applicable,Pasco,1,-10.6835926,-76.2561123


##### STOP HERE: Manually clean df_all_geo_clean.csv here. 
- Delete previous map.html file.
- Then re-run folium.

# 9. Export data mart sql versions
- df_single_with_grps merge with all df_exploded as list

In [4]:
# 9a. List of all exploded dfs
import pandas as pd
import os
from pathlib import Path
import numpy as np

# Retrieve all the exploded csv outputs
csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)


exploded_filepaths = [
    x for x in csv_outputs_dir.iterdir()
    if x.name.endswith("_exploded_cleaned.csv")
]

exploded_filepaths

[WindowsPath('csv_outputs_dir/ai_tools_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/attended_inperson_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/attended_online_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/busint_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/cloudplat_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/dep_init_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/digitools_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/generaltools_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/hardware_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/ingestion_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/orchest_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/otherfb_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/restofrole_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/spneeds_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/storage_exploded_cleaned.csv'),
 WindowsPath('csv_outputs_dir/succes

In [5]:
# 9b. Dictionary of all  df_exploded files
# Key is {col} file, value is a dataframe
dfs_exploded = {}

for path in exploded_filepaths:
    col = path.stem.replace("_exploded_cleaned", "")
    df = pd.read_csv(path)
    dfs_exploded[col] = df

dfs_exploded.keys()

dict_keys(['ai_tools', 'attended_inperson', 'attended_online', 'busint', 'cloudplat', 'dep_init', 'digitools', 'generaltools', 'hardware', 'ingestion', 'orchest', 'otherfb', 'restofrole', 'spneeds', 'storage', 'successmethod', 'transform', 'volunteer', 'warehs', 'whatused'])

In [6]:
dfs_exploded["ai_tools"].head()

,resp_id,ai_tools
0,1,Chatgpt
1,2,Chatgpt
2,2,Copilot
3,3,Chatgpt
4,3,Copilot


In [7]:
# 9c. Combine everything into duckdb with df_single_with_grps

import duckdb
con = duckdb.connect("survey2026.duckdb")

#  Store all multi-response dfs as tables
for name, df in dfs_exploded.items():
    con.register("tmp", df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM tmp")

# Store df_single_with_grps
df_single_with_grps = pd.read_csv(csv_outputs_dir/"df_single_with_grps.csv")
con.register("tmp", df_single_with_grps)
con.execute("CREATE OR REPLACE TABLE df_single_with_grps AS SELECT * FROM tmp")


# Retrieve the combined ph and non ph geo coded file 
location_dir = Path("location_dir")
location_dir.mkdir(exist_ok = True)

# Store df_all_geo_clean
df_all_geo_clean = pd.read_csv(location_dir/"df_all_geo_clean.csv")
con.register("tmp", df_all_geo_clean)
con.execute("CREATE OR REPLACE TABLE df_all_geo_clean AS SELECT * FROM tmp")

# Inspect all tables
con.execute("SHOW TABLES").df()



,name
0,ai_tools
1,attended_inperson
2,attended_online
3,busint
4,cloudplat
5,dep_init
6,df_all_geo_clean
7,df_single_with_grps
8,digitools
9,generaltools


In [8]:
# 9d. close the connection
con.close()

# Log
import datetime
print("Duckdb created. Connection closed.")
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Duckdb created. Connection closed.
Date and Time: 2026-03-27 09:53:10


In [9]:
#9e. Create a sqlite version for side explorations
# A lot smaller than duckdb since it has no metadata

import duckdb
con = duckdb.connect("survey2026.duckdb")

con.execute("DETACH DATABASE IF EXISTS sqlite_db")
con.execute("ATTACH 'survey2026.sqlite' AS sqlite_db (TYPE sqlite)")
tables = con.execute("SHOW TABLES").fetchall()
for (table_name,) in tables:
    con.execute(f"DROP TABLE IF EXISTS sqlite_db.{table_name}")
    con.execute(f"""
        CREATE TABLE sqlite_db.{table_name} AS
        SELECT * FROM {table_name}
    """)
con.close()

# Log
print("Sqlite created. Connection closed.")
print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Sqlite created. Connection closed.
Date and Time: 2026-03-27 09:54:05


# 10.  Export to staging and data_mart

In [10]:
#10a. Export excel file to data_mart staging
import pandas as pd
from pathlib import Path
import shutil

# Directories
csv_outputs_dir = Path("csv_outputs_dir")
location_dir = Path("location_dir")

data_mart_staging_excel_dir = Path("data_mart_staging/excel")
data_mart_staging_excel_dir.mkdir(parents=True,exist_ok=True)

# 1. Identify final csv files
single_file = csv_outputs_dir / "df_single_with_grps.csv"
geo_file = location_dir / "df_all_geo_clean.csv"
exploded_files = list(csv_outputs_dir.glob("*_exploded_cleaned.csv"))
summary_single = csv_outputs_dir / "summary_single_with_grps.csv"
summary_multi = csv_outputs_dir/ "summary_multi_response.csv"

files_to_copy = [single_file, geo_file] + exploded_files + [summary_single, summary_multi]

# 2. Copy final csv files to data_mart_staging
for f in files_to_copy:
    if f.exists():
        shutil.copy(f, data_mart_staging_excel_dir / f.name)
    else:
        print(f"Missing file:", f)

# 3. Build Excel workbook from the final CSVs
xlsx_path = data_mart_staging_excel_dir / "for_tableau.xlsx"

with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    for f in files_to_copy:
        if f.exists():
            df = pd.read_csv(f)
            sheet_name = f.stem[:31]  # Excel sheet name limit
            df.to_excel(writer, sheet_name=sheet_name, index=False)

print("for_tableau.xlsx created in data_mart_staging_excel_dir.")
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


for_tableau.xlsx created in data_mart_staging_excel_dir.
Date and Time:  2026-03-27 09:55:54


In [11]:
# 10a. Copy duckdb and sqlite files from cwd to data_mart_staging
# from pathlib import Path
# import shutil
# import datetime

cwd = Path.cwd()

def copy_by_extension(ext: str, target_subdir: str):
    target = Path("data_mart_staging") / target_subdir
    target.mkdir(parents=True, exist_ok=True)

    for f in cwd.glob(f"*.{ext}"):
        shutil.copy2(f, target / f.name)

    print(f"{ext} copied from cwd to staging.")
    print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


copy_by_extension("duckdb", "duckdb")
copy_by_extension("sqlite", "sqlite")


duckdb copied from cwd to staging.
Date and Time: 2026-03-27 09:56:00
sqlite copied from cwd to staging.
Date and Time: 2026-03-27 09:56:00


In [12]:
# 10b. Promote all data_mart_staging to data_mart
from pathlib import Path
import shutil

# Artifact configuration
ARTIFACTS = {
    "excel": "*.xlsx",
    "duckdb": "*.duckdb",
    "sqlite": "*.sqlite",
}

# Define parent roots
staging_root = Path("data_mart_staging")
data_mart_root = Path("data_mart")

# Ensure sub-directory structure exists
for artifact in ARTIFACTS:
    (staging_root / artifact).mkdir(parents=True, exist_ok=True)
    (data_mart_root / artifact).mkdir(parents=True, exist_ok=True)

# Promote a single artifact
def promote_artifact(artifact: str, pattern: str):
    staging_dir = staging_root / artifact
    target_dir = data_mart_root / artifact

    for f in staging_dir.glob(pattern):
        shutil.copy2(f, target_dir / f.name)

    print (f"{artifact} promoted from staging to data_mart.")
    print("Date and Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# Promote all artifacts
for artifact, pattern in ARTIFACTS.items():
    promote_artifact(artifact, pattern)


excel promoted from staging to data_mart.
Date and Time: 2026-03-27 09:56:10
duckdb promoted from staging to data_mart.
Date and Time: 2026-03-27 09:56:10
sqlite promoted from staging to data_mart.
Date and Time: 2026-03-27 09:56:10


In [13]:
# Export all files that will be used for summaries and crosstabs
import os
import shutil
import glob
import datetime
from pathlib import Path

# Define source file
csv_outputs_dir = Path("csv_outputs_dir")
csv_outputs_dir.mkdir(exist_ok=True)

# Copy of all the final outputs for data model
final_outputs_dir = Path("final_outputs_dir")
final_outputs_dir.mkdir(exist_ok=True)

# Copy files matching *_exploded_cleaned.csv
source_pattern1 = os.path.join(csv_outputs_dir, "*_exploded_cleaned.csv")
for src_file in glob.glob(source_pattern1):
    shutil.copy2(src_file, final_outputs_dir)

# Copy specific files
copy_files = ["df_single_with_grps.csv", "df_multi.csv", "summary_single_with_grps.csv", "summary_multi_response.csv"]

for file_name in copy_files:
    source_path = os.path.join(csv_outputs_dir, file_name)
    if os.path.exists(source_path):
        shutil.copy2(source_path, final_outputs_dir)

print ("final_outputs_dir created and populated.")
print ("Date and Time: ", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

final_outputs_dir created and populated.
Date and Time:  2026-03-27 10:02:13


# 11. Cleaning after cross-tabs

In [43]:
df_single_with_grps = pd.read_parquet(parquet_outputs_dir/"df_single_with_grps.parquet")
resp_id_chg_region = [742, 1005, 1086, 1207, 1486, 1562, 1592]
mask_region = df_single_with_grps['resp_id'].isin(resp_id_chg_region)
# check city_ph of resp_id_chg_region
df_single_with_grps.loc[mask_region, ['resp_id', 'country','region_ph', 'city_ph']]

,resp_id,country,region_ph,city_ph
741,742,Canada,Not Applicable,Caloocan
1004,1005,UAE,Not Applicable,Aleosan
1085,1086,KSA,Not Applicable,Bulacan
1206,1207,United States,Not Applicable,Aleosan
1485,1486,UAE,Not Applicable,Bulacan
1561,1562,Vietnam,Not Applicable,Bulacan
1591,1592,India,Not Applicable,Taguig


# No need to edit.   All region_ph values are already Not Applicable even if city_ph has a valid city_ph.
- region_ph has already used 'country' == 'Philippines' as another filter.